## Setup

This is not one of the seven steps. It prepares the notebook before Step 1 begins.

Before the analysis starts, the notebook needs a place to keep its files and the tools it will use. This cell does three things:

1. **Loads two helper modules:** `os` lets Python work with files and folders. `json` lets Python read and write JSON files, which are used to store the prompt structure (Step 4), the extracted phrases (Step 5) and the counts (Step 6).
2. **Creates five folders:** One for each kind of output, so the project stays organised by step.

| Folder | What it holds | Step |
|---|---|---|
| `corpus` | The full text of the story | 1 and 3 |
| `partial_reading` | The three blocks read in the partial reading | 1 |
| `prompt` | The prompt written for the language model | 4 |
| `extraction` | The model's output, the verified phrases and the counts | 5 and 6 |
| `docs` | The charts, the interpretation and the final report | 6 and 7 |

3. **Installs three Python libraries:** `plotly` for the interactive charts, `scipy` for the statistical tests (chi-square and Spearman correlation), and `pandas` for the tables. Colab usually has them already, so this is a safeguard.

Creating the folders first means every later step can save its output without errors, and anyone who reruns the notebook gets the same folder structure. Files in Colab are deleted when the session ends, so the last cell of the notebook zips the folders for download.

In [ ]:
import os, json
for folder in ["corpus", "partial_reading", "prompt", "extraction", "docs"]:
    os.makedirs(folder, exist_ok=True)

!pip install -q plotly scipy pandas
print("Setup done")

Setup done


# STEP 1 — Partial Reading

This partial reading serves as a sample rather than a summary. First, the story is divided into three equal parts based on the total sentence count. From each part, a specific block equal to 20% of that section is extracted: the first block of Part 1, a random block from Part 2, and the last block of Part 3. By using a fixed random seed (42), the exact same random block is chosen every time, ensuring that the entire sampling process can be easily reproduced and reported.

## Code explanation
It loads the story, splits it into sentences, divides the sentences into three equal parts, and picks one short block of sentences from each part (first, random, last) so that a partial reading can be done.

### 1. Loading the tools

`import re, random`

- `re` is Python's module for pattern matching in text ("regular expressions"). It is used to split the story into sentences.
- `random` is Python's module for random choices. It is used to pick the random block.
- `os` was already loaded in the Setup cell, so it is not imported again.

### 2. Defining where the story is stored

`STORY_PATH = "corpus/gift_of_the_magi.txt"`

A variable that holds the file location, so the path is typed once and reused. Capital letters mark a setting that stays fixed.

### 3. The function that loads the story

```python
def load_story():
```

A function is a named block of code that can be run when needed. This one returns the story text. It works in three steps:

| Line | Meaning |
|---|---|
| `if not os.path.exists(STORY_PATH):` | Checks whether the story file is already in the `corpus` folder. |
| `uploaded = files.upload()` | If the file is missing, Colab opens an upload button. The file `gift_of_the_magi.txt` is chosen from the computer. |
| `open(STORY_PATH, "wb")` and `f.write(...)` | The uploaded file is saved inside the `corpus` folder. `"wb"` means "write in binary", which copies the file exactly as it is. |
| `return open(STORY_PATH, encoding="utf-8").read().strip()` | The file is opened and read as text. `utf-8` makes sure curly quotes and dashes are read correctly. `.strip()` removes empty space at the very beginning and end. |

`raw_story = load_story()` runs the function and stores the whole story in the variable `raw_story`.

### 4. Fixing the random choice

`random.seed(42)`

A seed is a starting number for the random generator. With the same seed, the "random" block is the same every time the notebook runs. This makes the sampling reproducible and reportable ("random block chosen with seed 42").

### 5. Splitting the story into sentences

`sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+|\n+", raw_story) if s.strip()]`

This line is read from the inside out:

- `re.split(r"(?<=[.!?])\s+|\n+", raw_story)` cuts the text at two kinds of places:
  - `(?<=[.!?])\s+` means "any space that comes right after a full stop, question mark or exclamation mark". These are the ends of sentences.
  - `\n+` means "any line break". Paragraph breaks therefore also cut the text.
- `s.strip()` removes extra spaces around each piece.
- `if s.strip()` throws away empty pieces.

The result `sents` is a list of sentences in story order. For this story it contains about 157 sentences.

*Known limitation:* the splitter is simple. It also cuts after abbreviations such as "Mr." and "Mrs.", so a few sentences are split in two. This does not affect the research, because the same list is used for every part.

### 6. Dividing the sentences into three equal parts

```python
third = len(sents) // 3
parts = [sents[:third], sents[third:2 * third], sents[2 * third:]]
```

- `len(sents)` is the number of sentences (157).
- `// 3` is division that drops the remainder: 157 // 3 = 52.
- `parts` is a list of three lists:
  - Part 1 = sentences 1 to 52,
  - Part 2 = sentences 53 to 104,
  - Part 3 = sentences 105 to 157 (53 sentences, because the leftover sentence goes to the last part).

This matches the rule: *divide the text into equal parts.*

### 7. Deciding how long each block is

`n = max(3, round(0.2 * third))`

- `0.2 * third` = 20% of one part = 0.2 × 52 = 10.4.
- `round(...)` rounds it to **10**.
- `max(3, ...)` is a safety rule: the block is never shorter than 3 sentences, even for a very short text.

So each block has **10 sentences**, which is 20% of a part. This matches the rule: *read 20% of each part.*

### 8. Choosing where each block starts

`starts = [0, random.randint(0, len(parts[1]) - n), len(parts[2]) - n]`

This list holds three starting positions (counted from 0):

| Part | Start | Meaning |
|---|---|---|
| Part 1 | `0` | The block starts at the very first sentence, so the **first** 10 sentences are read. |
| Part 2 | `random.randint(0, len(parts[1]) - n)` | A random whole number between 0 and 42. The block starts at that position, so a **random** 10 sentences in a row are read. Because of the seed, this number is the same every run (40 for the 2,057-word story, which means sentences 41 to 50 of Part 2). |
| Part 3 | `len(parts[2]) - n` | 53 − 10 = 43. The block starts 10 sentences before the end, so the **last** 10 sentences are read. |

The `- n` in the random choice stops the block from running past the end of the part.

`labels = ["first", "random", "last"]` stores the names used when printing.

### 9. Building the blocks

`blocks = [" ".join(parts[i][starts[i]:starts[i] + n]) for i in range(3)]`

For each part (0, 1, 2):
- `parts[i][starts[i]:starts[i] + n]` cuts out the 10 sentences that start at the chosen position.
- `" ".join(...)` glues them back into one piece of text with spaces between the sentences.

`blocks` is therefore a list of three texts: the beginning, a random middle, and the ending.

### 10. Printing the result

```python
print(f"{len(sents)} sentences; each part has about {third}; each block has {n} sentences.\n")
for i, b in enumerate(blocks):
    print(f"--- PART {i + 1} ({labels[i]} block, sentences {starts[i] + 1}-{starts[i] + n} of the part) ---")
    print(b, "\n")
```

- The first `print` shows the totals (157 sentences, about 52 per part, 10 per block).
- The loop goes through the three blocks. `enumerate` gives each block a number (0, 1, 2) and the text. `i + 1` turns 0, 1, 2 into Part 1, 2, 3, and `starts[i] + 1` turns the Python position (counted from 0) into a normal sentence number (counted from 1).
- The output is the three blocks that are read for the initial impression in Step 2.

### Summary of the logic

| Stage | Result |
|---|---|
| Load | Whole story in `raw_story` |
| Split | About 157 sentences in `sents` |
| Divide | 3 equal parts (52, 52, 53 sentences) |
| Block size | 20% of a part = 10 sentences |
| Positions | Part 1 first, Part 2 random (seed 42), Part 3 last |
| Output | Three blocks of 10 sentences each, printed for reading |

In [ ]:
import re, random

STORY_PATH = "corpus/gift_of_the_magi.txt"

def load_story():
    """Use corpus/gift_of_the_magi.txt if it exists; otherwise ask for an upload (Colab)."""
    if not os.path.exists(STORY_PATH):
        from google.colab import files
        print("Upload gift_of_the_magi.txt now.")
        uploaded = files.upload()
        name = next(iter(uploaded))
        with open(STORY_PATH, "wb") as f:
            f.write(uploaded[name])
    return open(STORY_PATH, encoding="utf-8").read().strip()

raw_story = load_story()

random.seed(42)                       # keeps the random middle block the same every run
sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+|\n+", raw_story) if s.strip()]
third = len(sents) // 3
parts = [sents[:third], sents[third:2 * third], sents[2 * third:]]
n = max(3, round(0.2 * third))        # block length = 20 % of one part
starts = [0, random.randint(0, len(parts[1]) - n), len(parts[2]) - n]
labels = ["first", "random", "last"]

blocks = [" ".join(parts[i][starts[i]:starts[i] + n]) for i in range(3)]
print(f"{len(sents)} sentences; each part has about {third}; each block has {n} sentences.\n")
for i, b in enumerate(blocks):
    print(f"--- PART {i + 1} ({labels[i]} block, sentences {starts[i] + 1}-{starts[i] + n} of the part) ---")
    print(b, "\n")

157 sentences; each part has about 52; each block has 10 sentences.

--- PART 1 (first block, sentences 1-10 of the part) ---
One dollar and eighty-seven cents. That was all. And sixty cents of it was in pennies. Pennies saved one and two at a time by bulldozing the grocer and the vegetable man and the butcher until one’s cheeks burned with the silent imputation of parsimony that such close dealing implied. Three times Della counted it. One dollar and eighty-seven cents. And the next day would be Christmas. There was clearly nothing to do but flop down on the shabby little couch and howl. So Della did it. Which instigates the moral reflection that life is made up of sobs, sniffles, and smiles, with sniffles predominating. 

--- PART 2 (random block, sentences 41-50 of the part) ---
Jim stopped inside the door, as immovable as a setter at the scent of quail. His eyes were fixed upon Della, and there was an expression in them that she could not read, and it terrified her. It was not ange

### Step 1b — The phrases I marked while reading
During the reading of the three blocks, phrases that prompted a response are underlined. Rather than serving as data, these marked sections function as raw material for Step 2. A verification script then checks each underlined phrase to ensure it appears exactly as written within its corresponding block.

## Code explanation: Step 1b, recording the marked phrases

**What this cell does in one sentence.** It stores the phrases that stood out during the partial reading, checks that each one really appears in the block it came from, and saves the three blocks together with the marked phrases in a text file.

**Why this cell exists.** A partial reading is only useful if what was noticed can be shown. The marked phrases are the reader's underlining: the words that caused a reaction. They are not data for counting. They are the raw material for Step 2, where they are turned into an impression and then into dimensions. The check in this cell makes sure the notes cannot drift away from the text (a mistyped or half-remembered phrase is caught immediately).

**Requirement from the previous cell.** This cell uses the variables `blocks` and `labels`, which are created in the Step 1 code cell. That cell must be run first.

### 1. The marked phrases

```python
MARKED = {
    1: [...],
    2: [...],
    3: [...],
}
```

`MARKED` is a dictionary: a collection of labelled entries. Each key (1, 2, 3) is a part number, and each value is a list of phrases noticed in that part's block.

| Part | Block read | Phrases marked | What they show |
|---|---|---|---|
| 1 | First block (the opening) | "pennies", "cheeks burned with the silent imputation of parsimony", "shabby little couch", "sobs, sniffles, and smiles" | Money counted in small coins, and shame felt in the body |
| 2 | Random block | "it terrified her", "I had my hair cut off and sold", "My hair grows awfully fast" | The loss is already done, and it is spoken of lightly and anxiously |
| 3 | Last block (the ending) | "I sold the watch to get the money to buy your combs", "wise men", "two foolish children", "the wisest", "They are the magi" | The second sale, and the narrator's renaming of foolishness as wisdom |

The phrases are copied exactly as they appear in the story, including commas and capital letters.

### 2. The normalising function

```python
norm_q = lambda s: s.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"').lower()
```

A `lambda` is a short, one-line function. This one, named `norm_q`, prepares a piece of text for comparison by doing two things:

1. **Turns curly quotation marks into straight ones.** The story uses curly marks (’ ‘ “ ”), but they are easily typed as straight marks (' "). Without this step, "My hair grows awfully fast" might fail to match only because of a different quotation mark or apostrophe.
2. **Turns everything into lowercase** (`.lower()`), so that capital letters do not cause a mismatch ("The wisest" and "the wisest" count as the same).

### 3. The safety check

```python
for i, phrases in MARKED.items():
    for p in phrases:
        assert norm_q(p) in norm_q(blocks[i - 1]), f"Marked phrase not in block {i}: {p}"
```

Read from the outside in:

- `for i, phrases in MARKED.items()` goes through each part: `i` is the part number (1, 2, 3) and `phrases` is its list.
- `for p in phrases` goes through each phrase in that list.
- `blocks[i - 1]` is the block text of that part. The `- 1` is needed because Python counts list positions from 0, while parts are numbered from 1.
- `norm_q(p) in norm_q(blocks[i - 1])` asks: "after normalising, does the phrase occur inside the block?"
- `assert` stops the program with an error message if the answer is "no".

**What the output means:**

| Result | Meaning |
|---|---|
| Nothing is printed and the cell goes on | Every marked phrase was found in its block. |
| `AssertionError: Marked phrase not in block 2: ...` | That phrase is not in the block of that part. Either the phrase was copied incorrectly, or the sample changed (for example a different seed or a different block length), so the phrase now belongs to a part that was not read. The phrase must be corrected or replaced by one that appears in the printed block. |

### 4. Writing the methodology note

```python
methodology = (
    "SAMPLING METHODOLOGY\n"
    "I split the story ... "
    "one-fifth (20%) ... "
    "the last block of Part 3.\n\n"
)
```

Several strings placed one after another inside brackets are joined by Python into one long string. `\n` means "start a new line", so `\n\n` leaves a blank line. This text records how the sample was taken: three equal parts by sentence count, one block of about one-fifth (20%) from each part, taken from the first, a random (seed 42) and the last position.

### 5. Building the output text

```python
out = methodology + "MARKED PHRASES\n"
for i in range(3):
    out += f"\nPART {i + 1} ({labels[i]})\n{blocks[i]}\n\nMarked: " + "; ".join(MARKED[i + 1]) + "\n"
```

- `out` starts as the methodology note followed by the heading "MARKED PHRASES".
- `for i in range(3)` repeats three times (i = 0, 1, 2).
- In every round, the f-string adds:
  - the heading `PART 1 (first)`, `PART 2 (random)` or `PART 3 (last)` (`labels[i]` supplies the word in brackets),
  - the full text of the block (`blocks[i]`),
  - the line `Marked:` followed by the phrases of that part, separated by semicolons (`"; ".join(...)`).
- `+=` means "add to what is already in `out`".

The finished file looks like this:

```
SAMPLING METHODOLOGY
(how the sample was taken)

MARKED PHRASES

PART 1 (first)
(the 10 sentences)

Marked: pennies; cheeks burned with ...; ...

PART 2 (random)
...
```

### 6. Saving the file

```python
open("partial_reading/partial_reading.txt", "w", encoding="utf-8").write(out)
print("Saved partial_reading/partial_reading.txt")
```

- `open(..., "w")` opens the file for writing. If the file exists, it is replaced.
- `encoding="utf-8"` keeps the special characters (curly quotes, dashes) intact.
- `.write(out)` puts the text into the file.
- The file is saved in the `partial_reading` folder created in the Setup cell.
- The `print` line confirms that the cell finished.

### Summary

| Stage | What happens |
|---|---|
| Record | The phrases noticed in each block are stored in `MARKED` |
| Check | Each phrase is searched in its block (ignoring capitals and quote style); any mismatch stops the cell with an error |
| Document | A methodology note, the three blocks and the marked phrases are put together in one text |
| Save | The text is saved as `partial_reading/partial_reading.txt` |

**Link to the next step.** The phrases marked here (money and shame in Part 1, the sold hair in Part 2, the sold watch and "wise" in Part 3) are what the Step 2 impression is built from. The four dimensions of Step 2 (transactional shaming, corporeal commodification, objects and feeling, paradoxical value inversion) can each be traced back to these notes.

In [ ]:
MARKED = {
    1: ["pennies", "cheeks burned with the silent imputation of parsimony",
        "shabby little couch", "sobs, sniffles, and smiles"],
    2: ["it terrified her", "I had my hair cut off and sold",
        "My hair grows awfully fast"],
    3: ["I sold the watch to get the money to buy your combs", "wise men",
        "two foolish children", "the wisest", "They are the magi"],
}

norm_q = lambda s: s.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"').lower()
for i, phrases in MARKED.items():
    for p in phrases:
        assert norm_q(p) in norm_q(blocks[i - 1]), f"Marked phrase not in block {i}: {p}"

methodology = (
    "SAMPLING METHODOLOGY\n"
    "I split the story into three equal parts by sentence count. From each part I read one block of about "
    "one-fifth (20%) of the part: the first block of Part 1, a randomly placed block of Part 2 (seed 42) and "
    "the last block of Part 3.\n\n"
)
out = methodology + "MARKED PHRASES\n"
for i in range(3):
    out += f"\nPART {i + 1} ({labels[i]})\n{blocks[i]}\n\nMarked: " + "; ".join(MARKED[i + 1]) + "\n"
open("partial_reading/partial_reading.txt", "w", encoding="utf-8").write(out)
print("Saved partial_reading/partial_reading.txt")

Saved partial_reading/partial_reading.txt


# STEP 2: Initial Impression

**What the step requires:** a reader-oriented approach, in which the reader's own situation allows the impression to be received. This is a phenomenography: the meaning of a literary text appears when it is read by a reader.

**What is done in this step.** After the partial reading, the first understanding of the story is written down before any extraction or counting. It has six parts:

1. **Sampling methodology:** how the three blocks were chosen (Step 1).
2. **Impression:** what the story seemed to be about after reading the blocks.
3. **One overall concept:** a single idea that joins the marked phrases. Here it is *value inversion*, where loss is turned into emotional value.
4. **Positionality:** who is reading and from where, because the reader's background shapes what is noticed and what may be missed.
5. **Research problem:** the impression turned into a question and a claim to be tested.
6. **Dimensions:** four analytic categories, each with three sub-types and a definition: transactional shaming, corporeal commodification, objects and feeling, and paradoxical value inversion.

**What the code does.** It stores the dimensions, the research question and the claim in variables, builds the impression text, saves it as `initial_impression.md`, and displays it.

**Why it matters.** The impression is the base for the rest of the project. The dimensions set up here are used to write the prompt in Step 4, to decide what the language model extracts in Step 5, and to organise the charts and interpretation in Steps 6 and 7.

## Code explanation: Step 2, building and saving the initial impression

**What this cell does in one sentence.** It stores the four dimensions, the research question and the claim in variables, assembles them with the written impression into one formatted document, saves that document as `initial_impression.md`, and displays it in the notebook.

**Why it is one cell.** The impression is partly written text and partly generated text. The dimensions, question and claim are defined once as variables, so the same wording is reused here and in Step 4 (the prompt). No definition has to be typed twice, and the two steps cannot disagree.

### 1. Loading the display tools

```python
from IPython.display import Markdown, display
```

- `Markdown` marks a piece of text as formatted text (headings, bold, lists) instead of plain text.
- `display` shows that formatted text in the notebook output.

Together they make the impression appear with real headings and bold words, instead of showing symbols such as `##` and `**`.

### 2. The dimensions dictionary

```python
DIMENSIONS = {
    "transactional_shaming": {...},
    "corporeal_commodification": {...},
    "objects_and_feeling": {...},
    "paradoxical_value_inversion": {...},
}
```

`DIMENSIONS` is a dictionary inside a dictionary (a nested dictionary). It works like a filing cabinet:

| Level | Example | Meaning |
|---|---|---|
| Outer key | `"transactional_shaming"` | The code name of a dimension (no spaces, used by the program) |
| `"label"` | `"Transactional shaming"` | The readable name (used in headings and charts) |
| `"definition"` | `"Money handled as a source of exposure and shame..."` | One sentence explaining the dimension |
| `"subs"` | `{...}` | The three sub-types, each with its own definition |

**The four dimensions and their sub-types:**

| Dimension | Sub-types |
|---|---|
| 1. Transactional shaming | `amounts_money`, `counting_saving_bargaining`, `shame_and_want` |
| 2. Corporeal commodification | `body_as_sellable`, `price_sale_exchange`, `reaction_to_sale` |
| 3. Objects and feeling | `gifts_possessions`, `qualities_of_objects`, `feelings_attached` |
| 4. Paradoxical value inversion | `wisdom_foolishness`, `narrator_comments`, `ironic_register` |

This gives 4 dimensions × 3 sub-types = **12 categories**. These 12 categories are what the language model looks for in Step 5 and what the charts show in Step 6.

**Why the structure matters.** Later cells loop through this dictionary to build the prompt (Step 4), to check the model's answer (Step 5) and to label the figures (Step 6). If a dimension is renamed or a sub-type is added here, the later steps adjust automatically.

### 3. The research question and the claim

```python
RESEARCH_QUESTION = ("How does the narrator's language turn economic loss into emotional value, "
                     "and what does the reader learn to call \"wise\"?")
CLAIM = ("Words of shame, counting and selling are concentrated in the first two stages ...")
```

- `RESEARCH_QUESTION` is the question the whole project tries to answer.
- `CLAIM` is a statement that can be tested against the evidence. It is checked in Step 7, where it is found to be only partly supported.
- Text split over two lines inside brackets is joined by Python into one string.
- `\"` places a double quotation mark inside a string that is itself surrounded by double quotation marks.

### 4. Turning the dimensions into a readable list

```python
dim_lines = ""
for i, (k, d) in enumerate(DIMENSIONS.items(), 1):
    dim_lines += f"{i}. **{d['label']}.** {d['definition']}\n"
    for j, (s, text_) in enumerate(d["subs"].items()):
        dim_lines += f"   ({'abc'[j]}) *{s}*: {text_}\n"
```

This block builds a numbered list of the dimensions as text. It starts with an empty string `dim_lines` and adds to it, piece by piece.

**Outer loop (one round per dimension):**
- `DIMENSIONS.items()` gives each entry as a pair: the code name `k` and the details `d`.
- `enumerate(..., 1)` adds a counter `i` that starts at 1 (not 0), so the list is numbered 1, 2, 3, 4.
- The line added looks like `1. **Transactional shaming.** Money handled as ...`. The `**` marks make the label bold in Markdown. `d['label']` and `d['definition']` pull the readable name and the definition out of the dictionary.

**Inner loop (one round per sub-type):**
- `d["subs"].items()` gives each sub-type as a pair: the code name `s` and its definition `text_`. (The name `text_` has an underscore so that it does not overwrite the story text, which is stored in a variable called `text` in Step 3.)
- `enumerate` gives a counter `j` starting at 0.
- `'abc'[j]` picks a letter from the string "abc" by position: 0 gives a, 1 gives b, 2 gives c. This produces the labels (a), (b), (c).
- `*{s}*` puts the sub-type name in italics.
- `\n` ends the line, so the next item starts on a new line.

**Result (shortened):**

```
1. **Transactional shaming.** Money handled as a source of exposure and shame...
   (a) *amounts_money*: Sums of money, coins and wages ...
   (b) *counting_saving_bargaining*: Acts of counting, saving ...
   (c) *shame_and_want*: Words and phrases of shame ...
2. **Corporeal commodification.** ...
```

### 5. Assembling the impression

```python
IMPRESSION = f"""# Initial impression: ...
## Sampling methodology
...
## My dimensions
{dim_lines}"""
```

`IMPRESSION` is one long text made with an **f-string**. Two features are used:

- **Triple quotation marks (`"""`)** allow a text to run over many lines and to contain ordinary quotation marks without special handling.
- **The letter `f` and curly brackets `{...}`** let variables be inserted into the text. `{RESEARCH_QUESTION}`, `{CLAIM}` and `{dim_lines}` are replaced by their current values when the cell runs.

**The text lines start with `#` and `##`.** In Markdown, `#` marks a main heading and `##` marks a sub-heading. The document therefore has this structure:

| Heading | Content | Source in the code |
|---|---|---|
| Initial impression (title) | Title and story details | Typed text |
| Sampling methodology | How the three blocks were chosen | Typed text (matches Step 1) |
| Impression | What the story seemed to be about after the partial reading | Typed text |
| One overall concept | Value inversion | Typed text |
| Positionality | Who is reading and from where | Typed text |
| Research problem | The question and the claim | Typed text with `{RESEARCH_QUESTION}` and `{CLAIM}` inserted |
| Dimensions | The numbered list of the 4 dimensions and 12 sub-types | `{dim_lines}` inserted |

The impression and positionality are the reader's own words and are the parts that are meant to be personal. The question, claim and dimensions are inserted from the variables defined earlier in the same cell.

### 6. Saving and displaying

```python
open("initial_impression.md", "w", encoding="utf-8").write(IMPRESSION)
display(Markdown(IMPRESSION))
```

- `open("initial_impression.md", "w", ...)` opens (or creates) the file for writing. If the file already exists, it is replaced. `.md` is the extension for Markdown files.
- `encoding="utf-8"` keeps special characters, such as the curly apostrophe in "Jim’s" and the dash in the impression, intact.
- `.write(IMPRESSION)` puts the whole text into the file.
- `Markdown(IMPRESSION)` converts the text into formatted content, and `display(...)` shows it below the cell, with headings, bold words and the numbered list.

### Summary

| Stage | What happens |
|---|---|
| Define | The 4 dimensions (12 sub-types), the research question and the claim are stored in variables |
| Format | A loop turns the dimensions into a numbered, bold and italic list |
| Assemble | The typed impression and the inserted variables are combined in one f-string |
| Save | The text is written to `initial_impression.md` |
| Display | The formatted result is shown in the notebook |

**Link to later steps.**
- Step 4 reuses `DIMENSIONS` and `RESEARCH_QUESTION` to write the prompt.
- Step 5 uses `DIMENSIONS` to check that the model's output contains only valid dimensions and sub-types.
- Step 6 uses `DIMENSIONS` for the chart labels.
- Step 7 tests the `CLAIM` against the results.

For that reason, this cell has to be run before Steps 3 to 7.

In [ ]:
from IPython.display import Markdown, display

DIMENSIONS = {
    "transactional_shaming": {
        "label": "Transactional shaming",
        "definition": "Money handled as a source of exposure and shame, not only as a means of buying.",
        "subs": {
            "amounts_money": "Sums of money, coins and wages (for example dollars, cents, pennies).",
            "counting_saving_bargaining": "Acts of counting, saving, calculating, bargaining and planning to buy.",
            "shame_and_want": "Words and phrases of shame, shabbiness, poverty and lack.",
        },
    },
    "corporeal_commodification": {
        "label": "Corporeal commodification",
        "definition": "A part of the body treated as a thing that can be priced, sold and exchanged.",
        "subs": {
            "body_as_sellable": "Hair and the words used for it when it is treated as goods.",
            "price_sale_exchange": "Words of price, sale and exchange attached to the transaction.",
            "reaction_to_sale": "The body's and the self's reactions to the sale (fear, panic, self-image).",
        },
    },
    "objects_and_feeling": {
        "label": "Objects and feeling",
        "definition": "Gifts and possessions as carriers of feeling and character.",
        "subs": {
            "gifts_possessions": "The gifts and possessions themselves (watch, chain, combs, presents).",
            "qualities_of_objects": "Qualities given to objects (beautiful, rare, precious, simple and chaste).",
            "feelings_attached": "Feelings that the characters attach to people and objects.",
        },
    },
    "paradoxical_value_inversion": {
        "label": "Paradoxical value inversion",
        "definition": "The narrator turns loss into gain and foolishness into wisdom.",
        "subs": {
            "wisdom_foolishness": "Words of wisdom, foolishness, reason and calculation.",
            "narrator_comments": "The narrator's direct addresses, asides and comments on the telling.",
            "ironic_register": "Mock-heroic, formal or exaggerated phrases that create irony.",
        },
    },
}

RESEARCH_QUESTION = ("How does the narrator's language turn economic loss into emotional value, "
                     "and what does the reader learn to call \"wise\"?")
CLAIM = ("Words of shame, counting and selling are concentrated in the first two stages of the story, "
         "while words of wisdom and the narrator's comments are concentrated in the last stage.")

dim_lines = ""
for i, (k, d) in enumerate(DIMENSIONS.items(), 1):
    dim_lines += f"{i}. **{d['label']}.** {d['definition']}\n"
    for j, (s, text_) in enumerate(d["subs"].items()):
        dim_lines += f"   ({'abc'[j]}) *{s}*: {text_}\n"

IMPRESSION = f"""# Initial impression: "The Gift of the Magi" (O. Henry, 1905)

## Sampling methodology
The story is split into three equal parts. From each part, one block equal to about 20% of that section is read: the first block of Part 1, a random block from Part 2 using a fixed seed (42), and the last block of Part 3. Together, these three blocks cover the opening, the moment when Jim sees Della, and the ending.

## My impression
A reading of the three blocks reveals a tension between love and money, guided by a narrator who constantly addresses the audience. In the first block, money is counted repeatedly—emphasised by phrases like "One dollar and eighty-seven cents" and "pennies"—and is directly linked to physical reactions, causing cheeks to burn and prompting Della to howl. By the second block, the sacrifice has already occurred. Della does not describe the event as a loss, choosing instead to declare that her hair grows quickly, while Jim’s unreadable expression frightens her. In the final block, Jim reveals his own sale, prompting the narrator to shift from a joking tone to a direct address. The narrator labels the couple first as "foolish" and ultimately as "the wisest." This shift marks a distinct turning point, where the very acts that initially resembled poverty and waste are renamed as true wealth.

## One overall concept
The core concept uniting the marked phrases is value inversion, where the narrative language transforms a material loss into emotional value. Through this lens, sacrifice reads as richness, and apparent foolishness is reinterpreted as deep wisdom.

## My positionality
The perspective shaping this analysis is that of a college student from a busy, multilingual Indian city. While academic coursework is conducted entirely in English, it functions as an acquired register rather than the primary language spoken at home. This background naturally draws immediate attention to plot progression and emotional resonance, while rare or highly formal vocabulary is processed secondarily. Consequently, the interpretation remains highly alert to themes of financial constraint and familial obligation within a story about gift-giving, while the finer, ironic nuances of a formal English narrator may initially be less apparent.

## My research problem
Reading from the perspective of a student from an urban, multilingual Indian background, the analysis reveals that the first block treats money as a cause of shame, the second treats the body as a commodity that can be sold, and the third allows the narrator to redefine the nature of wisdom. This framework leads to the central inquiry: {RESEARCH_QUESTION} Consequently, the core thesis to be tested is: {CLAIM}

## My dimensions
{dim_lines}"""

open("initial_impression.md", "w", encoding="utf-8").write(IMPRESSION)
display(Markdown(IMPRESSION))

# Initial impression: "The Gift of the Magi" (O. Henry, 1905)

## Sampling methodology
The story is split into three equal parts. From each part, one block equal to about 20% of that section is read: the first block of Part 1, a random block from Part 2 using a fixed seed (42), and the last block of Part 3. Together, these three blocks cover the opening, the moment when Jim sees Della, and the ending.

## My impression
A reading of the three blocks reveals a tension between love and money, guided by a narrator who constantly addresses the audience. In the first block, money is counted repeatedly—emphasised by phrases like "One dollar and eighty-seven cents" and "pennies"—and is directly linked to physical reactions, causing cheeks to burn and prompting Della to howl. By the second block, the sacrifice has already occurred. Della does not describe the event as a loss, choosing instead to declare that her hair grows quickly, while Jim’s unreadable expression frightens her. In the final block, Jim reveals his own sale, prompting the narrator to shift from a joking tone to a direct address. The narrator labels the couple first as "foolish" and ultimately as "the wisest." This shift marks a distinct turning point, where the very acts that initially resembled poverty and waste are renamed as true wealth.

## One overall concept
The core concept uniting the marked phrases is value inversion, where the narrative language transforms a material loss into emotional value. Through this lens, sacrifice reads as richness, and apparent foolishness is reinterpreted as deep wisdom.

## My positionality
The perspective shaping this analysis is that of a college student from a busy, multilingual Indian city. While academic coursework is conducted entirely in English, it functions as an acquired register rather than the primary language spoken at home. This background naturally draws immediate attention to plot progression and emotional resonance, while rare or highly formal vocabulary is processed secondarily. Consequently, the interpretation remains highly alert to themes of financial constraint and familial obligation within a story about gift-giving, while the finer, ironic nuances of a formal English narrator may initially be less apparent.

## My research problem
Reading from the perspective of a student from an urban, multilingual Indian background, the analysis reveals that the first block treats money as a cause of shame, the second treats the body as a commodity that can be sold, and the third allows the narrator to redefine the nature of wisdom. This framework leads to the central inquiry: How does the narrator's language turn economic loss into emotional value, and what does the reader learn to call "wise"? Consequently, the core thesis to be tested is: Words of shame, counting and selling are concentrated in the first two stages of the story, while words of wisdom and the narrator's comments are concentrated in the last stage.

## My dimensions
1. **Transactional shaming.** Money handled as a source of exposure and shame, not only as a means of buying.
   (a) *amounts_money*: Sums of money, coins and wages (for example dollars, cents, pennies).
   (b) *counting_saving_bargaining*: Acts of counting, saving, calculating, bargaining and planning to buy.
   (c) *shame_and_want*: Words and phrases of shame, shabbiness, poverty and lack.
2. **Corporeal commodification.** A part of the body treated as a thing that can be priced, sold and exchanged.
   (a) *body_as_sellable*: Hair and the words used for it when it is treated as goods.
   (b) *price_sale_exchange*: Words of price, sale and exchange attached to the transaction.
   (c) *reaction_to_sale*: The body's and the self's reactions to the sale (fear, panic, self-image).
3. **Objects and feeling.** Gifts and possessions as carriers of feeling and character.
   (a) *gifts_possessions*: The gifts and possessions themselves (watch, chain, combs, presents).
   (b) *qualities_of_objects*: Qualities given to objects (beautiful, rare, precious, simple and chaste).
   (c) *feelings_attached*: Feelings that the characters attach to people and objects.
4. **Paradoxical value inversion.** The narrator turns loss into gain and foolishness into wisdom.
   (a) *wisdom_foolishness*: Words of wisdom, foolishness, reason and calculation.
   (b) *narrator_comments*: The narrator's direct addresses, asides and comments on the telling.
   (c) *ironic_register*: Mock-heroic, formal or exaggerated phrases that create irony.


# STEP 3 — Text Upload

Pre-processing can be performed for removing noise. Because the model must receive the whole story, this initial step loads and cleans the full text. The cleaning process remains deliberately light to preserve the integrity of the literary text: carriage returns are removed, repeated spaces and tabs are collapsed, sequences of three or more blank lines are reduced to a single line, and non-narrative lines—such as page numbers, web headers, and footers—are stripped out. Curly quotes, dashes, and original spellings are kept exactly as they appear. Following the clean-up, the code verifies the data structure by checking the total word count (2,057), confirming the first and last words, and validating the paragraph count (45), with these paragraph numbers subsequently used for segmenting the story during Step 6.

## Code explanation: Step 3, loading, cleaning and checking the full story

**What this cell does in one sentence.** It reads the full story, removes any lines that are not part of it (page numbers, web addresses, print headers), tidies the spacing without touching the wording, checks that the text is complete, and shows a table of its 45 paragraphs.

**Requirements from earlier cells.** The cell uses `re` and `STORY_PATH`, which are defined in the Step 1 code cell, so that cell has to be run first.

### 1. Loading pandas

```python
import pandas as pd
```

`pandas` is a library for working with tables (similar to a spreadsheet). Here it is used at the end of the cell to display the paragraph table. `pd` is a short nickname, so `pandas.DataFrame` can be written as `pd.DataFrame`.

### 2. Reading the story

```python
text = open(STORY_PATH, encoding="utf-8").read()
text = text.replace("\r", "")
```

- The first line opens the story file and reads all of it into the variable `text`. `encoding="utf-8"` makes sure that curly quotation marks and long dashes are read correctly.
- `"\r"` is a hidden "carriage return" character that Windows adds at the end of lines. Removing it prevents hidden characters from interfering with later steps.

### 3. Removing noise lines

```python
noise = re.compile(r"^\s*(page\s+\d+(\s+of\s+\d+)?|\d+\s*/\s*\d+|https?://\S+|\d{1,2}/\d{1,2}/\d{2,4},?.*)\s*$", re.I)
text = "\n".join(line for line in text.split("\n") if not noise.match(line))
```

**What "noise" means.** When a text is copied from a website or a PDF, extra lines often come along that are not part of the story. This cell removes them.

**The pattern (`noise`).** It is a regular expression: a short code that describes a type of text. A line is treated as noise only if the **whole line** matches one of four shapes:

| Part of the pattern | Matches | Example |
|---|---|---|
| `page\s+\d+(\s+of\s+\d+)?` | A page label | `Page 3` or `Page 3 of 10` |
| `\d+\s*/\s*\d+` | A page counter | `4/17` or `4 / 17` |
| `https?://\S+` | A line that is only a web address | `https://colab.research.google.com/...` |
| `\d{1,2}/\d{1,2}/\d{2,4},?.*` | A date stamp, as added by browsers when printing | `10/31/26, 12:39 PM ...` |

- `^` and `$` mean "start of the line" and "end of the line", so only complete noise lines are caught. A story sentence that merely contains a number is not touched.
- `re.I` makes the pattern ignore capital letters ("PAGE 3" is caught too).

**The filter.** `text.split("\n")` cuts the text into lines. The generator `line for line in ... if not noise.match(line)` keeps only the lines that are **not** noise. `"\n".join(...)` glues the kept lines back into one text.

For the clean story file used here, no lines are removed. The step is a safeguard for texts copied from other sources.

### 4. Tidying the spacing

```python
text = re.sub(r"[ \t]+", " ", text)
text = re.sub(r"\n{3,}", "\n\n", text).strip()
```

| Line | What it does |
|---|---|
| `re.sub(r"[ \t]+", " ", text)` | Replaces any run of spaces or tabs with one single space. |
| `re.sub(r"\n{3,}", "\n\n", text)` | Replaces three or more line breaks in a row (several blank lines) with exactly one blank line. |
| `.strip()` | Removes empty space at the very beginning and end of the text. |

**What is deliberately not changed.** Curly quotation marks, dashes, spelling, capital letters and punctuation stay exactly as written. A literary text should not be altered, because style is part of what is being studied.

### 5. Saving the cleaned story

```python
open(STORY_PATH, "w", encoding="utf-8").write(text)
```

The cleaned text is written back to the same file (`"w"` means "write", replacing the old content). From now on, the file in the `corpus` folder is the cleaned version. Running the cell again does no harm, because cleaning an already clean text changes nothing.

### 6. Splitting the story into paragraphs

```python
paras = [re.sub(r"\s*\n\s*", " ", p).strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
```

Read from the inside out:

- `re.split(r"\n\s*\n", text)` cuts the text wherever there is a **blank line** (a line break, optional spaces, another line break). Each piece is one paragraph.
- `re.sub(r"\s*\n\s*", " ", p)` replaces any remaining line breaks inside a paragraph with a single space, so each paragraph becomes one continuous line.
- `.strip()` removes extra space at the ends.
- `if p.strip()` skips empty pieces.

The result `paras` is a list of paragraphs. For this story it has 45 items. This is why the story file must have a blank line between paragraphs: without blank lines, the whole story would count as one paragraph.

### 7. Printing a quick report

```python
print(len(text.split()), "words;", len(paras), "paragraphs")
print("START:", text[:60])
print("END:  ", text[-40:])
```

- `text.split()` cuts the text at spaces, and `len(...)` counts the pieces, which gives the number of words.
- `len(paras)` counts the paragraphs.
- `text[:60]` shows the first 60 characters, and `text[-40:]` shows the last 40.

**Expected output for the complete story:**

```
2057 words; 45 paragraphs
START: One dollar and eighty-seven cents. That was all. And sixty c
END:   here they are wisest. They are the magi.
```

If the numbers or the start and end differ, the file is incomplete or was pasted with changes.

### 8. The safety checks

```python
assert len(text.split()) > 1500, "The story is missing or incomplete. Upload the file again."
assert len(paras) >= 20, "Line breaks were lost. Each paragraph must be separated by a blank line."
```

`assert` stops the cell with an error message if a condition is false.

| Check | Protects against |
|---|---|
| More than 1,500 words | A missing, empty or half-pasted story. A story of about 2,000 words passes easily. |
| At least 20 paragraphs | A paste in which the blank lines between paragraphs were lost. |

These checks prevent an incomplete text from reaching the extraction (Step 5), where it would produce wrong counts without any warning.

### 9. The paragraph table

```python
para_table = pd.DataFrame({"paragraph": range(1, len(paras) + 1),
                           "words": [len(p.split()) for p in paras],
                           "begins": [p[:70] for p in paras]})
para_table
```

A `DataFrame` is a table. It has three columns:

| Column | Contents |
|---|---|
| `paragraph` | The paragraph number, 1 to 45 (`range(1, len(paras) + 1)` starts at 1, not 0) |
| `words` | The number of words in each paragraph |
| `begins` | The first 70 characters of each paragraph, so each one can be recognised |

The last line, `para_table`, shows the table under the cell. (Colab displays the value of the last line of a cell.)

**Why the table matters.** The paragraph numbers are used in Step 6 (`SEGMENT_STARTS`) to divide the story into scenes. For example, the table shows that paragraph 12 is where Della reaches Madame Sofronie's shop and paragraph 45 is the narrator's closing paragraph about the magi.

### Summary

| Stage | What happens |
|---|---|
| Read | The story file is loaded into `text` |
| Remove noise | Page numbers, web addresses and print headers are deleted |
| Tidy | Extra spaces and blank lines are reduced; wording and punctuation are untouched |
| Save | The cleaned text replaces the file in `corpus/` |
| Split | The story is divided into 45 paragraphs |
| Check | Word count and paragraph count are tested; the cell stops if the text looks incomplete |
| Show | A table lists each paragraph with its number, length and opening words |

**Link to later steps.** The variable `text` is used in Step 4 (the full prompt) and in Step 5 (checking that every extracted phrase occurs in the story). The list `paras` is used in Step 6 (segmentation).

In [ ]:
import pandas as pd

text = open(STORY_PATH, encoding="utf-8").read()
text = text.replace("\r", "")

# remove obvious noise lines (page numbers, "Page x of y", copy-paste headers and footers)
noise = re.compile(r"^\s*(page\s+\d+(\s+of\s+\d+)?|\d+\s*/\s*\d+|https?://\S+|\d{1,2}/\d{1,2}/\d{2,4},?.*)\s*$", re.I)
text = "\n".join(line for line in text.split("\n") if not noise.match(line))

text = re.sub(r"[ \t]+", " ", text)
text = re.sub(r"\n{3,}", "\n\n", text).strip()
open(STORY_PATH, "w", encoding="utf-8").write(text)

paras = [re.sub(r"\s*\n\s*", " ", p).strip() for p in re.split(r"\n\s*\n", text) if p.strip()]

print(len(text.split()), "words;", len(paras), "paragraphs")
print("START:", text[:60])
print("END:  ", text[-40:])
assert len(text.split()) > 1500, "The story is missing or incomplete. Upload the file again."
assert len(paras) >= 20, "Line breaks were lost. Each paragraph must be separated by a blank line."

para_table = pd.DataFrame({"paragraph": range(1, len(paras) + 1),
                           "words": [len(p.split()) for p in paras],
                           "begins": [p[:70] for p in paras]})
para_table

2057 words; 45 paragraphs
START: One dollar and eighty-seven cents. That was all. And sixty c
END:   here they are wisest. They are the magi.


,paragraph,words,begins
0,1,67,One dollar and eighty-seven cents. That was al...
1,2,38,There was clearly nothing to do but flop down ...
2,3,48,While the mistress of the home is gradually su...
3,4,39,In the vestibule below was a letter-box into w...
4,5,80,The “Dillingham” had been flung to the breeze ...
5,6,127,Della finished her cry and attended to her che...
6,7,56,There was a pier glass between the windows of ...
7,8,40,Suddenly she whirled from the window and stood...
8,9,106,"Now, there were two possessions of the James D..."
9,10,60,So now Della’s beautiful hair fell about her r...


# STEP 4 — Query Generation

the query is developed from the initial impression; this is a part of prompt engineering, so prompt-engineering skill is needed.

The query is built *directly from the dimensions of Step 2*, so the extraction answers my research problem and not a generic question. The prompt has five parts:

1. **A role and an objective** (the research question).
2. **Definitions** of every sub-type, taken from Step 2.
3. **Strict rules for evidence:** copy phrases exactly (so they can be verified), put each phrase under one sub-type only (so nothing is counted twice), list each phrase once (counting is done by code), and avoid generic words (for example *took*, *table*, *chops*).
4. **A fixed output format:** JSON with the exact keys of my dimensions.
5. **A placeholder** `{{PASSAGE}}` that is replaced by the full story.

The code builds the prompt from `DIMENSIONS`, checks that the JSON structure is valid, and writes the full prompt ready to paste into an LLM.

## Code explanation: Step 4, building the extraction prompt

**What this cell does in one sentence.** It writes the instructions (the prompt) that will be given to a language model, builds them from the dimensions defined in Step 2, checks that the required answer format is valid, and saves two files: the prompt with an empty slot for the story, and the full prompt with the story inserted.

**Requirements from earlier cells.** The cell uses `json` (Setup), `DIMENSIONS` and `RESEARCH_QUESTION` (Step 2) and `text` (Step 3). Those cells have to be run first.

### 1. Building the answer format (the schema)

```python
schema = {d: {s: [] for s in info["subs"]} for d, info in DIMENSIONS.items()}
schema_text = json.dumps(schema, indent=1, ensure_ascii=False)
```

**The schema** is an empty answer sheet. The model is asked to fill it in, so its answer can be read by code.

- The line is a *dictionary comprehension* (a short way to build a dictionary with a loop). For each dimension `d`, it creates an entry; inside it, for each sub-type `s`, it creates an empty list `[]`.
- The result has the same shape as `DIMENSIONS`, but with empty lists instead of definitions:

```
{
 "transactional_shaming": {
  "amounts_money": [],
  "counting_saving_bargaining": [],
  "shame_and_want": []
 },
 "corporeal_commodification": { ... },
 "objects_and_feeling": { ... },
 "paradoxical_value_inversion": { ... }
}
```

- `json.dumps(...)` turns this Python dictionary into JSON text, a standard format for structured data.
- `indent=1` adds line breaks and one space of indentation, so the text is easy to read.
- `ensure_ascii=False` keeps any special characters as they are.

Because the schema is generated from `DIMENSIONS`, its keys always match the dimensions of Step 2. They cannot drift apart.

### 2. Writing the definitions

```python
def_lines = ""
for d, info in DIMENSIONS.items():
    def_lines += f"\n{info['label']} ({d}): {info['definition']}\n"
    for s, text_ in info["subs"].items():
        def_lines += f"  - {s}: {text_}\n"
```

This loop builds a block of text that tells the model what each category means. It starts with an empty string and adds to it:

- Outer loop: one line per dimension, with its readable name, code name and definition.
- Inner loop: one bullet line per sub-type, with its code name and definition.
- `\n` means "new line".
- The variable is called `text_` (with an underscore) so that it does not overwrite `text`, which holds the story.

The result looks like this:

```
Transactional shaming (transactional_shaming): Money handled as a source of exposure and shame...
  - amounts_money: Sums of money, coins and wages ...
  - counting_saving_bargaining: Acts of counting, saving ...
  - shame_and_want: Words and phrases of shame ...
```

### 3. Assembling the prompt

```python
PROMPT = f"""You are assisting a literary researcher ... {RESEARCH_QUESTION} ... {def_lines} ... {schema_text} ... PASSAGE: {{{{PASSAGE}}}}"""
```

The prompt is one long f-string (a text in which `{variable}` is replaced by the variable's value). It has the five parts of a well-built prompt:

| Part | Text in the prompt | Purpose |
|---|---|---|
| 1. Role and objective | "You are assisting a literary researcher..." and the research question | Tells the model what the work is for, so extractions are targeted and not generic |
| 2. Task | "Read the PASSAGE below. Extract the words and phrases..." | States exactly what to do |
| 3. Definitions | `{def_lines}` | Explains each of the 12 categories |
| 4. Rules | Seven numbered rules (see below) | Controls the quality of the evidence |
| 5. Output format and text | `{schema_text}` and `PASSAGE:` with a placeholder | Fixes the answer format and marks where the story goes |

**The seven rules and the reason for each:**

| Rule | Instruction | Why it is needed |
|---|---|---|
| 1 | Copy every item exactly; no paraphrasing | Models sometimes reword or invent quotations. Exact copies can be checked word for word in Step 5. |
| 2 | Choose the shortest phrase, but keep idioms whole | Gives precise evidence ("close dealing" stays together) without long sentences. |
| 3 | Each phrase under one sub-type only | Prevents the same words from being counted in two categories. |
| 4 | Each phrase listed once | Counting is done by code in Step 6, which is more accurate than counting by a model. |
| 5 | No ordinary words (took, was, table, chops) | Common words would inflate the counts and weaken the analysis. |
| 6 | Single words only when central (hair, sold, wisest) | Allows the key vocabulary while blocking vague one-word entries. |
| 7 | Return only valid JSON, no commentary | The answer can be loaded by code without cleaning. |

**The strange-looking `{{{{PASSAGE}}}}`.** In an f-string, doubled curly brackets `{{` and `}}` produce a single literal bracket. Four on each side therefore produce `{{PASSAGE}}` in the final text. That placeholder marks the slot where the story will be inserted.

### 4. Saving the prompt

```python
open("prompt/prompt.txt", "w", encoding="utf-8").write(PROMPT)
```

The prompt, with the empty `{{PASSAGE}}` slot, is saved in the `prompt` folder. `"w"` means "write" (an existing file is replaced), and `encoding="utf-8"` keeps special characters intact. This file is the record of the query as designed, without the story attached.

### 5. Validating the prompt

```python
block = PROMPT[PROMPT.index("{\n"):PROMPT.index("PASSAGE:")]
json.loads(block)
print("JSON structure is valid. Placeholder present:", "{{PASSAGE}}" in PROMPT)
```

- `PROMPT.index("{\n")` finds the position where the JSON block starts (the first opening bracket followed by a line break).
- `PROMPT.index("PASSAGE:")` finds the position of the final "PASSAGE:" label.
- `PROMPT[start:end]` cuts out the text between them, which is the schema.
- `json.loads(block)` tries to read that text as JSON. If the structure were broken (for example a missing bracket or comma), this line would stop the cell with an error.
- The `print` line also checks that the placeholder `{{PASSAGE}}` is in the prompt. The expected output is:

```
JSON structure is valid. Placeholder present: True
```

### 6. Creating the full prompt

```python
full_prompt = PROMPT.replace("{{PASSAGE}}", text)
open("extraction/full_prompt.txt", "w", encoding="utf-8").write(full_prompt)
print(f"\nfull_prompt.txt: {len(full_prompt.split())} words\n")
print(PROMPT)
```

- `PROMPT.replace("{{PASSAGE}}", text)` swaps the placeholder for the full story, so the instructions and the story are now one text.
- The result is saved as `extraction/full_prompt.txt`. This is the file to copy into a language model (for example Claude or Gemini) in Step 5.
- The word count printed is the length of the instructions plus the 2,057 words of the story.
- The last line prints the prompt with the empty placeholder, so the instructions can be read on screen without the whole story filling the output.

### Summary

| Stage | Result |
|---|---|
| Schema | An empty answer sheet with the 12 categories, generated from `DIMENSIONS` |
| Definitions | A text block explaining each category |
| Prompt | Role, task, definitions, seven rules, answer format and a placeholder for the story |
| Save | `prompt/prompt.txt` (instructions with an empty slot) |
| Validate | The JSON structure is tested and the placeholder is checked |
| Full prompt | The story is inserted and saved as `extraction/full_prompt.txt` |

**Link to the other steps.** The query comes straight from the initial impression: the research question and the four dimensions of Step 2 are written into it, so the extraction answers the research problem and not a generic question. The model's answer to this prompt is stored and checked in Step 5. The rules in this step (exact copies, one sub-type per phrase, no common words) are what make the verification in Step 5 and the counting in Step 6 possible.

In [ ]:
schema = {d: {s: [] for s in info["subs"]} for d, info in DIMENSIONS.items()}
schema_text = json.dumps(schema, indent=1, ensure_ascii=False)

def_lines = ""
for d, info in DIMENSIONS.items():
    def_lines += f"\n{info['label']} ({d}): {info['definition']}\n"
    for s, text_ in info["subs"].items():
        def_lines += f"  - {s}: {text_}\n"

PROMPT = f"""You are assisting a literary researcher who is doing a reader-oriented analysis of O. Henry's short story "The Gift of the Magi" (1905).

RESEARCH QUESTION: {RESEARCH_QUESTION}

TASK: Read the PASSAGE below. Extract the words and phrases in which the narrator's language expresses each sub-type defined here.
{def_lines}
RULES:
1. Copy every item exactly as it appears in the passage. Do not paraphrase, correct or add words.
2. Choose the shortest phrase that carries the meaning, but keep idioms and collocations whole (for example "close dealing", "silent imputation of parsimony").
3. Put each distinct phrase under ONE sub-type only. If it could fit two, choose the better fit.
4. List each distinct phrase once. Do not repeat it; the researcher's code counts the occurrences.
5. Do not extract ordinary function words or generic verbs and nouns (for example took, was, table, chops).
6. A single word is allowed only when it is central to the dimension (for example hair, sold, wisest).
7. Return ONLY valid JSON with exactly this structure, with no commentary and no markdown fences.

{schema_text}

PASSAGE:
{{{{PASSAGE}}}}
"""
open("prompt/prompt.txt", "w", encoding="utf-8").write(PROMPT)

# validation: the JSON block is valid and the placeholder is present
block = PROMPT[PROMPT.index("{\n"):PROMPT.index("PASSAGE:")]
json.loads(block)
print("JSON structure is valid. Placeholder present:", "{{PASSAGE}}" in PROMPT)

# the full prompt, ready to paste into an LLM
full_prompt = PROMPT.replace("{{PASSAGE}}", text)
open("extraction/full_prompt.txt", "w", encoding="utf-8").write(full_prompt)
print(f"\nfull_prompt.txt: {len(full_prompt.split())} words\n")
print(PROMPT)

JSON structure is valid. Placeholder present: True

full_prompt.txt: 2497 words

You are assisting a literary researcher who is doing a reader-oriented analysis of O. Henry's short story "The Gift of the Magi" (1905).

RESEARCH QUESTION: How does the narrator's language turn economic loss into emotional value, and what does the reader learn to call "wise"?

TASK: Read the PASSAGE below. Extract the words and phrases in which the narrator's language expresses each sub-type defined here.

Transactional shaming (transactional_shaming): Money handled as a source of exposure and shame, not only as a means of buying.
  - amounts_money: Sums of money, coins and wages (for example dollars, cents, pennies).
  - counting_saving_bargaining: Acts of counting, saving, calculating, bargaining and planning to buy.
  - shame_and_want: Words and phrases of shame, shabbiness, poverty and lack.

Corporeal commodification (corporeal_commodification): A part of the body treated as a thing that can be price

# STEP 5 — Targeted Extraction and Formatting

depending on the research objective, the extraction is targeted; this part relies on the LLM's pattern-recognition capacity.

The model is not asked to summarise the story, only to find phrases that belong to my twelve sub-types. Four moves:

1. **Extraction.** The full prompt from Step 4 (`extraction/full_prompt.txt`) was given to a language model, which returned the JSON in the next cell. To repeat this with another model (for example Gemini), paste `full_prompt.txt` into it, replace the JSON with the model's answer, and change `MODEL_NAME` two cells later.
2. **Verification.** A model can invent or rewrite quotations, so code checks every phrase: it must occur word for word in the story, and under only one sub-type. Otherwise it is removed.
3. **Researcher review.** I read the verified list and may cut phrases by adding them to `RESEARCHER_REJECT`. Every removal is printed with its reason.
4. **Formatting.** The verified phrases are saved as `verified_items.json` and `Theme_JSON.py`. A README records the model, date and prompt.

**Matching rules used in Step 6.** Phrases are matched on whole words, case-insensitively, longest phrase first. Once a stretch of text is matched, no shorter phrase can match it again, so nothing is counted twice and no words are counted in two dimensions. A short stoplist removes false friends (for example "at present" meaning *now*, not *gift*).

## Code explanation: Step 5a, storing the language model's output

**What this cell does in one sentence.** It stores the model's answer to the Step 4 prompt, a list of 152 words and phrases sorted into the 12 sub-types, and saves it as `extraction/llm_output.json`.

**What this cell does not do.** It performs no analysis and no checking. It only keeps the answer exactly as it was received. Checking happens in the next cell (Step 5b).

### 1. Where the content comes from

The prompt built in Step 4 (`extraction/full_prompt.txt`) was given to a language model. The model returned the JSON text stored in this cell. This is the step that relies on the model's pattern-recognition ability: finding, across the whole story, the wording that belongs to each sub-type.

- The output stored here was produced by **Claude (Anthropic)**, and `MODEL_NAME` in the next cell records this.
- To repeat the extraction with another model (for example Gemini), paste `full_prompt.txt` into that model, copy its answer, and replace everything between the triple quotation marks with it. Then change `MODEL_NAME` in the next cell, so that the record states the model actually used.

### 2. The variable and the raw string

```python
LLM_OUTPUT = r"""
{ ... }
"""
```

- `LLM_OUTPUT` is a variable that holds the answer as text.
- The `r` before the quotation marks makes it a **raw string**: backslashes and special symbols are kept exactly as typed, with no hidden changes.
- The triple quotation marks (`"""`) allow the text to run over many lines and to contain ordinary quotation marks, which JSON uses a lot.

### 3. The structure of the answer

The text is in JSON format, with three levels. It follows the answer sheet (schema) defined in Step 4:

| Level | Example | Meaning |
|---|---|---|
| Dimension | `"transactional_shaming"` | One of the four dimensions from Step 2 |
| Sub-type | `"amounts_money"` | One of the three sub-types inside each dimension |
| List of phrases | `["pennies", "penny", "$1.87", ...]` | The words and phrases found in the story for that sub-type |

The model therefore returned 4 dimensions × 3 sub-types = 12 lists.

### 4. What was extracted

| Dimension | Sub-type | Phrases | Examples |
|---|---|---|---|
| Transactional shaming | amounts_money | 12 | "pennies", "$1.87", "twenty-one dollars" |
| | counting_saving_bargaining | 11 | "bulldozing", "close dealing", "counted" |
| | shame_and_want | 10 | "cheeks burned", "shabby", "mendicancy squad" |
| Corporeal commodification | body_as_sellable | 8 | "hair", "brown cascade", "tresses" |
| | price_sale_exchange | 6 | "sold", "i buy hair", "twenty dollars" |
| | reaction_to_sale | 15 | "panting", "a tear or two", "truant schoolboy" |
| Objects and feeling | gifts_possessions | 17 | "watch", "combs", "fob chain", "gifts" |
| | qualities_of_objects | 13 | "beautiful", "rare", "simple and chaste in design" |
| | feelings_attached | 20 | "mighty pride", "worshipped", "ecstatic scream of joy" |
| Paradoxical value inversion | wisdom_foolishness | 11 | "wise", "wisest", "foolish children", "magi" |
| | narrator_comments | 15 | "dear friends", "as you know", "poor fellow" |
| | ironic_register | 14 | "tripped by on rosy wings", "queen of sheba" |
| **Total** | | **152** | |

### 5. How the output follows the prompt rules

| Rule from the prompt (Step 4) | How it shows in this output |
|---|---|
| Copy exactly, no paraphrasing | Every item is a piece of wording taken from the story. In this file the items are written in lowercase, because the later steps compare text without regard to capital letters. |
| Keep idioms and collocations whole | Long phrases stay together: "silent imputation of parsimony", "ravages made by generosity added to love". |
| One sub-type per phrase | No phrase appears twice in the file. Related words are kept separate when they are different strings: "wise", "wise men", "wonderfully wise men". |
| No ordinary words | Words such as "took", "table" and "chops" are absent. |
| Single words only when central | One-word items are key vocabulary: "hair", "sold", "wisest", "magi", "watch", "combs". |

**Overlapping phrases.** Some entries contain others, for example "twenty dollars" and "twenty dollars a week", or "wise" and "wise men". This is intended. In Step 6 the longest phrase is matched first, and a stretch of text that has been matched cannot be matched again. So "twenty dollars a week" is counted once, as a wage in `amounts_money`, and does not also count as a sale price in `price_sale_exchange`.

### 6. Saving the answer

```python
open("extraction/llm_output.json", "w", encoding="utf-8").write(LLM_OUTPUT.strip())
print("Saved extraction/llm_output.json")
```

- `.strip()` removes the empty line that comes right after the opening quotation marks and before the closing ones.
- `open(..., "w", encoding="utf-8").write(...)` creates the file in the `extraction` folder and writes the text into it (replacing any older version). `utf-8` keeps special characters intact.
- The file is the model's answer **as received**. The next cell reads this file, checks it and creates the cleaned versions. Keeping the original untouched means the process can be inspected: anyone can compare what the model said with what was kept.

### Summary

| Stage | What happens |
|---|---|
| Source | The Step 4 prompt was answered by a language model |
| Content | 152 phrases in 12 lists (4 dimensions × 3 sub-types) |
| Format | JSON, matching the schema from Step 4 |
| Save | `extraction/llm_output.json`, unchanged from the model's answer |
| Next | Step 5b checks every phrase against the story, then formats the results |

**Link to the other steps.** The phrases stored here are the evidence base for the whole analysis: the counts and charts in Step 6 and the interpretation in Step 7 all come from this list. A language model can make mistakes or invent wording, so none of these phrases is trusted until Step 5b has verified it word for word against the story.

In [ ]:
LLM_OUTPUT = r"""
{
  "transactional_shaming": {
    "amounts_money": [
      "one dollar and eighty-seven cents", "a dollar and eighty-seven cents", "sixty cents",
      "pennies", "penny", "$1.87", "$8", "$30", "$20", "twenty-one dollars", "87 cents",
      "twenty dollars a week"
    ],
    "counting_saving_bargaining": [
      "bulldozing", "close dealing", "counted", "saved", "saving", "calculated", "contracting",
      "expenses had been greater", "doesn't go far", "to buy jim a present",
      "to buy a present for jim"
    ],
    "shame_and_want": [
      "cheeks burned", "silent imputation of parsimony", "shabby", "mendicancy squad",
      "worn red carpet", "on the sly", "burdened with a family", "needed a new overcoat",
      "without gloves", "howl"
    ]
  },
  "corporeal_commodification": {
    "body_as_sellable": [
      "hair", "hairs of my head", "brown cascade", "tresses",
      "lifting the mass with a practised hand", "hair goods of all kinds", "cut off",
      "cut it off"
    ],
    "price_sale_exchange": [
      "will you buy my hair", "i buy hair", "twenty dollars", "sold", "exchange",
      "i sold the watch to get the money to buy your combs"
    ],
    "reaction_to_sale": [
      "panting", "give it to me quick", "a tear or two", "faltered", "nervously",
      "lost its color", "turned white", "truant schoolboy", "coney island chorus girl",
      "ravages made by generosity added to love", "please god, make him think i am still pretty",
      "my hair grows so fast", "my hair grows awfully fast", "it'll grow out again",
      "if jim doesn't kill me"
    ]
  },
  "objects_and_feeling": {
    "gifts_possessions": [
      "gold watch", "watch", "the watch", "platinum fob chain", "fob chain", "chain",
      "leather strap", "combs", "present", "presents", "gift", "gifts", "treasures",
      "possessions", "pure tortoise shell", "jewelled rims", "coveted adornments"
    ],
    "qualities_of_objects": [
      "simple and chaste in design", "quietness and value", "value", "beautiful", "rare",
      "sterling", "fine", "expensive", "worthy", "dull precious metal", "too nice", "nice",
      "grand"
    ],
    "feelings_attached": [
      "mighty pride", "envy", "worshipped", "craved", "yearned", "ecstatic scream of joy",
      "hysterical tears and wails", "terrified", "sobs, sniffles, and smiles", "hugged", "happy",
      "smile", "smiled", "ardent spirit", "intoxication", "darling", "silent prayer", "anxious",
      "serious sweetness", "dim eyes"
    ]
  },
  "paradoxical_value_inversion": {
    "wisdom_foolishness": [
      "wise", "wise men", "wonderfully wise men", "wise ones", "wisest", "foolish children",
      "unwisely", "prudence and reason", "idiocy",
      "a mathematician or a wit would give you the wrong answer", "magi"
    ],
    "narrator_comments": [
      "dear friends", "let us regard with discreet scrutiny", "perhaps you have seen",
      "take a look at the home", "as you know", "forget the hashed metaphor",
      "this dark assertion will be illuminated later on", "which is all very good",
      "they always are", "which instigates the moral reflection",
      "here i have lamely related to you",
      "in a last word to the wise of these days let it be said",
      "which is always a tremendous task", "poor fellow", "already introduced to you"
    ],
    "ironic_register": [
      "did not exactly beggar description", "tripped by on rosy wings",
      "meretricious ornamentation", "inconsequential object", "uneventful chronicle",
      "the lord of the flat", "the mistress of the home", "queen of sheba", "king solomon",
      "her majesty's jewels and gifts", "eight dollars a week or a million a year",
      "nobody could ever count my love for you", "the greatest treasures of their house",
      "the magi brought valuable gifts, but that was not among them"
    ]
  }
}
"""
open("extraction/llm_output.json", "w", encoding="utf-8").write(LLM_OUTPUT.strip())
print("Saved extraction/llm_output.json")

Saved extraction/llm_output.json


### Step 5b — Verification, researcher review and formatting

## Code explanation: Step 5b, verifying, reviewing and formatting the extracted phrases

**What this cell does in one sentence.** It reads the model's answer, checks every phrase against the story, removes anything that fails, records every removal, and saves the verified phrases in four files: `verified_items.json`, `raw_llm_output.json`, `Theme_JSON.py` and `README.md`.

**Why this cell is needed.** A language model can invent a quotation, slightly reword one, or place the same phrase in two categories. Counting unchecked phrases would put errors into the charts and the interpretation. This cell is the quality gate between the model's answer and the analysis.

**Requirements from earlier cells.** The cell uses `re` and `json` (Setup and Step 1), `DIMENSIONS` (Step 2), `text` (Step 3) and `extraction/llm_output.json` (Step 5a). Those cells have to be run first.

### 1. The settings

```python
MODEL_NAME = "Claude (Anthropic), claude.ai chat"
EXTRACTION_DATE = "31 October 2026"
RESEARCHER_REJECT = []
STOPLIST = ["at present"]
```

| Setting | Purpose |
|---|---|
| `MODEL_NAME` | Records which model produced the extraction. Change it if another model is used (for example Gemini), because it appears in the saved files and the report. |
| `EXTRACTION_DATE` | Records when the extraction was done. Change it to the actual date. |
| `RESEARCHER_REJECT` | A list for phrases that the researcher decides to cut after reading the verified list. It is empty by default. Example: `["took", "fine"]`. This is the step of **researcher review**, in which the reader judges the machine's choices. |
| `STOPLIST` | "False friends": phrases that look relevant but mean something else. "at present" means *now* in the story ("too nice to use just at present"), not *gift*, although it contains the word "present". Stoplist phrases are matched in Step 6 so that the word inside them is not counted, but they are never counted themselves. |

### 2. Reading the model's answer

```python
raw_json = open("extraction/llm_output.json", encoding="utf-8").read().strip()
raw_json = re.sub(r"^```(?:json)?|```$", "", raw_json, flags=re.M).strip()
data = json.loads(raw_json)
```

- The first line reads the saved answer from Step 5a.
- The second line removes the "code fences": the three backtick marks that language models often put around JSON (sometimes with the word `json` after the opening ones). If an answer from another model is pasted with these marks, the line cleans them off, so the JSON can still be read. With the current file, nothing is removed.
- `json.loads(raw_json)` turns the JSON text into a Python dictionary called `data`. If the text is not valid JSON (for example a missing bracket), the cell stops here with an error.

### 3. The two helper tools

```python
norm = lambda s: " ".join(str(s).lower().replace("’", "'").replace("‘", "'")
                          .replace("“", '"').replace("”", '"').split())
word_re = lambda k: re.compile(r"(?<!\w)" + re.escape(k) + r"(?!\w)")
ntext = norm(text)
```

**`norm` prepares text for fair comparison.** It does four things:

| Action | Why |
|---|---|
| `.lower()` | Capital letters should not cause a mismatch ("The Watch" matches "the watch"). |
| Curly quotes and apostrophes to straight ones | The story uses ’ and “ ”, but a phrase may be typed with ' and ". |
| `.split()` then `" ".join(...)` | Any run of spaces or line breaks becomes a single space, so a phrase that runs over a line break still matches. |
| `str(s)` | Makes sure that the input is treated as text. |

**`word_re` builds a whole-word search pattern for a phrase.**

- `re.escape(k)` makes symbols such as `$` and `.` be read literally. Without this, the `$` in "$1.87" would be read as a pattern symbol and fail.
- `(?<!\w)` means "the character before the match must not be a letter or digit" and `(?!\w)` means "the character after must not be one either". Together they demand a **whole word**. For example, "hair" does not match inside "haircut", and "wise" does not match inside "wiser" or "otherwise".

**`ntext`** is the whole story after `norm`, ready to be searched.

### 4. Checking that the model used the right categories

```python
for d, subs in data.items():
    assert d in DIMENSIONS, f"Unknown dimension: {d}"
    for s in subs:
        assert s in DIMENSIONS[d]["subs"], f"Unknown sub-type: {d} / {s}"
```

Every dimension and sub-type name in the model's answer must exist in `DIMENSIONS` (Step 2). If the model invented a category or misspelled one, the cell stops with a message naming it. This keeps the answer in the same structure as the prompt asked for.

### 5. The main verification loop

```python
verified, removed, seen = {}, [], {}
for d, subs in data.items():
    for s, items in subs.items():
        for it in items:
            k = norm(it).strip(" .,;:!?\"'")
            if not k:
                continue
            if k in RESEARCHER_REJECT:
                removed.append((d, s, it, "rejected by researcher"))
            elif k in seen:
                removed.append((d, s, it, f"duplicate of {seen[k][0]} / {seen[k][1]}"))
            elif not word_re(k).search(ntext):
                removed.append((d, s, it, "not found word for word in the story"))
            else:
                seen[k] = (d, s)
                verified.setdefault(d, {}).setdefault(s, []).append(k)
```

**Three containers are created.**

| Name | Holds |
|---|---|
| `verified` | The phrases that pass every test, organised by dimension and sub-type |
| `removed` | A log of every phrase that fails, with the reason |
| `seen` | A lookup of phrases already accepted, with their category (used to catch duplicates) |

**For each phrase (three nested loops: dimension, sub-type, phrase):**

1. `k = norm(it).strip(...)` normalises the phrase and removes stray punctuation at its edges. `if not k: continue` skips empty entries.
2. The phrase then faces three tests, in this order. The first test it fails decides its fate:

| Test | Question | If failed |
|---|---|---|
| 1. Researcher review | Is the phrase in `RESEARCHER_REJECT`? | Removed: "rejected by researcher" |
| 2. Duplicate check | Was the same phrase already accepted under another sub-type? | Removed: "duplicate of ..." (so nothing is counted in two categories) |
| 3. Verbatim check | Does the phrase occur, as a whole word or phrase, in the story? | Removed: "not found word for word in the story" |

3. A phrase that passes all three is added to `seen` and to `verified`, under its dimension and sub-type. `setdefault` creates the dimension and sub-type entries the first time they are needed, so the code does not have to prepare them in advance.

**What the check proves, and what it does not.** It proves that every kept phrase really occurs in the story and is placed in one category only. It does **not** prove that the phrase belongs in that category. That judgement belongs to the researcher, which is the purpose of `RESEARCHER_REJECT`.

### 6. Saving the files

**a) `verified_items.json`**

```python
json.dump(verified, open("extraction/verified_items.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
```

The verified phrases in JSON, with `indent=2` for readability and `ensure_ascii=False` so special characters stay as they are.

**b) `raw_llm_output.json`**

The model's answer (`data`) is saved together with the model name, the date and the prompt file. The file records exactly **what the model said and who said it**, which is needed for citing and reproducing the extraction.

**c) `Theme_JSON.py`**

The code builds the lines of a small Python file by hand:

- Two comment lines (starting with `#`) state the source and the verification.
- `themes = {` opens a dictionary.
- For each sub-type, one key such as `"transactional_shaming: amounts_money": [` is written, followed by one line per phrase (`json.dumps` puts quotation marks around it) and a closing `],`.
- A final `}` closes the dictionary.

Step 6 loads this file and reads `themes` from it. A Python file is used because it can be read in one line, and because it can also be opened and read by a person.

**d) `README.md`**

An f-string creates a short record of the extraction: the model, the date, the prompt used, where the original output is saved, how verification worked, the researcher's rejections ("none" if the list is empty) and the matching rules used in Step 6 (whole words, longest phrase first, nothing counted twice, the stoplist). This note makes the extraction reproducible and can be quoted in the method section of the report.

### 7. The report on screen

```python
n_ver = sum(len(v) for subs in verified.values() for v in subs.values())
print(f"{n_ver} phrases verified; {len(removed)} removed.\n")
for d, subs in verified.items():
    for s, items in subs.items():
        print(f"{d} / {s}: {len(items)} phrases")
print("\nREMOVED:")
for r in removed:
    print("  ", r)
if not removed:
    print("   none")
```

- `n_ver` adds up the number of phrases in all the verified lists.
- The first print gives the total of verified and removed phrases.
- The loop prints how many phrases each of the 12 sub-types kept.
- The last part prints every removed phrase with its reason, or "none" if nothing was removed. Nothing is removed silently.

**Expected output for the stored extraction:**

```
152 phrases verified; 0 removed.

transactional_shaming / amounts_money: 12 phrases
transactional_shaming / counting_saving_bargaining: 11 phrases
transactional_shaming / shame_and_want: 10 phrases
corporeal_commodification / body_as_sellable: 8 phrases
...
REMOVED:
   none
```

If a different model's output is used, some phrases may be removed, and the list shows why. A removed phrase is usually a sign that the model reworded the text, which is the very problem this cell exists to catch.

### Summary

| Stage | What happens |
|---|---|
| Settings | Model, date, researcher rejections and stoplist are recorded |
| Read | The model's JSON answer is loaded (fences removed if present) |
| Structure check | Dimension and sub-type names must match Step 2 |
| Phrase checks | Researcher rejection, duplicate check, word-for-word check in the story |
| Log | Every removal is stored with its reason and printed |
| Save | `verified_items.json`, `raw_llm_output.json`, `Theme_JSON.py` and `README.md` |

**Link to the other steps.** `Theme_JSON.py` is the file that Step 6 reads to count the phrases in each segment of the story. The settings `STOPLIST` and `MODEL_NAME` are also used in Step 6 (matching) and Step 7 (the method note of the report). Everything counted or interpreted later therefore passes through this cell first.

In [ ]:
MODEL_NAME = "Claude (Anthropic), claude.ai chat"      # change this if you used another model
EXTRACTION_DATE = "31 October 2026"                     # change to your own date
RESEARCHER_REJECT = []                                   # phrases you decide to cut, e.g. ["took"]
STOPLIST = ["at present"]                                # false friends: matched but never counted

raw_json = open("extraction/llm_output.json", encoding="utf-8").read().strip()
raw_json = re.sub(r"^```(?:json)?|```$", "", raw_json, flags=re.M).strip()   # remove code fences if pasted
data = json.loads(raw_json)

norm = lambda s: " ".join(str(s).lower().replace("’", "'").replace("‘", "'")
                          .replace("“", '"').replace("”", '"').split())
ntext = norm(text)
word_re = lambda k: re.compile(r"(?<!\w)" + re.escape(k) + r"(?!\w)")

# the model must use exactly the dimensions and sub-types defined in Step 2
for d, subs in data.items():
    assert d in DIMENSIONS, f"Unknown dimension: {d}"
    for s in subs:
        assert s in DIMENSIONS[d]["subs"], f"Unknown sub-type: {d} / {s}"

verified, removed, seen = {}, [], {}
for d, subs in data.items():
    for s, items in subs.items():
        for it in items:
            k = norm(it).strip(" .,;:!?\"'")
            if not k:
                continue
            if k in RESEARCHER_REJECT:
                removed.append((d, s, it, "rejected by researcher"))
            elif k in seen:
                removed.append((d, s, it, f"duplicate of {seen[k][0]} / {seen[k][1]}"))
            elif not word_re(k).search(ntext):
                removed.append((d, s, it, "not found word for word in the story"))
            else:
                seen[k] = (d, s)
                verified.setdefault(d, {}).setdefault(s, []).append(k)

json.dump(verified, open("extraction/verified_items.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
json.dump({"model": MODEL_NAME, "date": EXTRACTION_DATE, "prompt_file": "prompt/prompt.txt",
           "merged": data}, open("extraction/raw_llm_output.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)

lines = ["# Words and phrases extracted from O. Henry's \"The Gift of the Magi\" for a synthetic reading.",
         f"# Extracted by: {MODEL_NAME}, {EXTRACTION_DATE}. Verified against the text and reviewed by the researcher.",
         "", "themes = {"]
for d, subs in verified.items():
    for s, items in subs.items():
        lines.append(f'    "{d}: {s}": [')
        lines += [f"        {json.dumps(x, ensure_ascii=False)}," for x in items]
        lines.append("    ],")
lines.append("}")
open("extraction/Theme_JSON.py", "w", encoding="utf-8").write("\n".join(lines))

readme = f"""# Extraction record
- Model: {MODEL_NAME}
- Date: {EXTRACTION_DATE}
- Prompt: prompt/prompt.txt (the full version sent is extraction/full_prompt.txt)
- Output as received: extraction/llm_output.json
- Verification: every phrase must occur word for word in the story and under one sub-type only.
- Researcher rejections: {RESEARCHER_REJECT if RESEARCHER_REJECT else "none"}
- Matching in Step 6: whole words, case-insensitive, longest phrase first, no stretch of text counted twice; stoplist: {STOPLIST}
"""
open("extraction/README.md", "w", encoding="utf-8").write(readme)

n_ver = sum(len(v) for subs in verified.values() for v in subs.values())
print(f"{n_ver} phrases verified; {len(removed)} removed.\n")
for d, subs in verified.items():
    for s, items in subs.items():
        print(f"{d} / {s}: {len(items)} phrases")
print("\nREMOVED:")
for r in removed:
    print("  ", r)
if not removed:
    print("   none")

152 phrases verified; 0 removed.

transactional_shaming / amounts_money: 12 phrases
transactional_shaming / counting_saving_bargaining: 11 phrases
transactional_shaming / shame_and_want: 10 phrases
corporeal_commodification / body_as_sellable: 8 phrases
corporeal_commodification / price_sale_exchange: 6 phrases
corporeal_commodification / reaction_to_sale: 15 phrases
objects_and_feeling / gifts_possessions: 17 phrases
objects_and_feeling / qualities_of_objects: 13 phrases
objects_and_feeling / feelings_attached: 20 phrases
paradoxical_value_inversion / wisdom_foolishness: 11 phrases
paradoxical_value_inversion / narrator_comments: 15 phrases
paradoxical_value_inversion / ironic_register: 14 phrases

REMOVED:
   none


# STEP 6 — Computation and Visualisation

Python or alternative third-party or self-developed tools can be used for visualisation.
The process consists of four parts:

6a. Segmentation

The story contains 45 paragraphs, which are divided into 13 segments based on changes of scene. These segments are then grouped into four stages:
• Stage I (Paragraphs 1–8): Scarcity and shame
• Stage II (Paragraphs 9–22): Sacrifice and exchange
• Stage III (Paragraphs 23–36): Confrontation
• Stage IV (Paragraphs 37–45): Revelation and revaluation
Scene-based segments are highly suited for examining narrative turning points, and their lengths remain relatively consistent, ranging from 80 to 223 words. To ensure fairness, all comparisons utilise a metric of hits per 100 words, preventing longer segments from appearing falsely prominent simply due to their word count.

6b. Counting

Each verified phrase is matched within each segment according to the rules established in Step 5. This operation generates a table featuring one row for every identified match.

6c. Statistics and Context

Analytical outputs include counts grouped by stage and sub-type, a chi-square test to determine if the distribution across stages deviates from expectations based purely on stage length, and a Spearman correlation to assess whether density changes steadily over time. Additionally, keyword-in-context lines are generated so that every statistical pattern can be cross-referenced against the actual sentences. Because hits within a single passage are not fully independent, these statistical tests serve as an analytical guide rather than formal mathematical proof.

6d. Visualisation

The analysis concludes with the generation of interactive figures.

### Step 6a — Segmentation

## Code explanation: Step 6a, dividing the story into segments and stages

**What this cell does in one sentence.** It cuts the 45 paragraphs of the story into 13 scenes (segments), groups those scenes into four stages, and stores for each segment its text, its length in words and the stage it belongs to.

**Why the story is divided.** The research question is about *where* in the story each kind of language appears. To answer it, the text has to be divided into units that can be compared. Cutting at scene changes is better than cutting into equal slices for a literary question, because the question is about turning points (the sale of the hair, the opening of the gifts, the narrator's final comment).

**Requirements from earlier cells.** The cell uses `paras` (Step 3) and `pd`, the pandas nickname (Step 3). That cell has to be run first.

### 1. The settings the researcher decides

```python
SEGMENT_STARTS = [1, 3, 6, 9, 12, 18, 20, 23, 27, 34, 37, 40, 45]
```

A list of the **paragraph numbers where each segment begins**. The first segment starts at paragraph 1, the second at paragraph 3, and so on. A segment runs from its start paragraph up to the paragraph before the next start. The last segment runs to the end of the story.

```python
STAGES = [(1, "I. Scarcity and shame"), (9, "II. Sacrifice and exchange"),
          (23, "III. Confrontation"), (37, "IV. Revelation and revaluation")]
```

A list of pairs: the paragraph where each **stage** begins, and the researcher's name for it. A stage is a group of neighbouring segments. Each stage must begin where a segment begins.

```python
SEGMENT_NOTES = ["Della counts $1.87 and cries", ... "The narrator's coda: the magi"]
```

A short description of each scene, one per segment. The notes appear in the table and in the hover text of the charts, so a reader can see what happens in each segment.

**The resulting division:**

| Segment | Paragraphs | Scene | Words | Stage |
|---|---|---|---|---|
| 1 | 1–2 | Della counts $1.87 and cries | 105 | I. Scarcity and shame |
| 2 | 3–5 | The flat and the Youngs' income | 167 | I |
| 3 | 6–8 | Della plans the present; the pier glass | 223 | I |
| 4 | 9–11 | The two treasures; the hair falls; Della leaves | 204 | II. Sacrifice and exchange |
| 5 | 12–17 | Madame Sofronie buys the hair | 80 | II |
| 6 | 18–19 | Shopping; the platinum chain | 179 | II |
| 7 | 20–22 | Repairs at home; her reflection | 121 | II |
| 8 | 23–26 | Waiting; Jim arrives | 210 | III. Confrontation |
| 9 | 27–33 | Della tells Jim she sold her hair | 207 | III |
| 10 | 34–36 | The narrator's aside; the package | 138 | III |
| 11 | 37–39 | The combs are opened | 150 | IV. Revelation and revaluation |
| 12 | 40–44 | The watch; Jim's confession | 151 | IV |
| 13 | 45 | The narrator's coda: the magi | 122 | IV |

The segments are not equal in length, because scenes are not equal in length. The lengths stay fairly close (80 to 223 words), so no single very long segment dominates the comparison. All later comparisons use *hits per 100 words*, which corrects for the remaining differences.

### 2. The safety checks

```python
assert SEGMENT_STARTS[0] == 1 and SEGMENT_STARTS == sorted(set(SEGMENT_STARTS))
assert STAGES[0][0] == 1 and all(s in SEGMENT_STARTS for s, _ in STAGES)
assert SEGMENT_STARTS[-1] <= len(paras) and len(SEGMENT_NOTES) == len(SEGMENT_STARTS)
```

`assert` stops the cell with an error if a condition is false. Together the three lines guard against typing mistakes in the settings:

| Check | What it prevents |
|---|---|
| The first segment starts at 1 | Leaving out the opening paragraphs |
| `SEGMENT_STARTS == sorted(set(SEGMENT_STARTS))` | Numbers out of order or repeated. `set` removes repeats and `sorted` puts the numbers in order. If the result differs from the original list, the original had a problem. |
| The first stage starts at 1 | A stage list that does not cover the whole story |
| Every stage start is also a segment start | A stage beginning in the middle of a segment |
| The last start is not larger than the number of paragraphs (45) | Pointing to a paragraph that does not exist |
| One note for each segment | A missing or extra description |

### 3. Building the segments

```python
bounds = SEGMENT_STARTS + [len(paras) + 1]
```

The start numbers plus one extra number, 46 (the number of paragraphs plus one). This closes the last segment: every segment ends where the next one begins, and the extra number plays the part of a "next segment" for the final one.

```python
segments = []
for k in range(len(SEGMENT_STARTS)):
    a, b = bounds[k], bounds[k + 1]
    seg_text = " ".join(paras[a - 1:b - 1])
    stage = [name for s, name in STAGES if s <= a][-1]
    segments.append({...})
```

The loop runs once for each segment (13 times). In each round:

| Line | What it does | Example: segment 5 |
|---|---|---|
| `a, b = bounds[k], bounds[k + 1]` | `a` is the first paragraph of the segment, `b` is the first paragraph of the next one. | a = 12, b = 18 |
| `paras[a - 1:b - 1]` | Cuts out the paragraphs from `a` up to but not including `b`. The `- 1` is needed because Python counts list positions from 0, while paragraphs are numbered from 1. | Paragraphs 12 to 17 |
| `" ".join(...)` | Glues those paragraphs into one text with spaces between them. | One text of 80 words |
| `stage = [name for s, name in STAGES if s <= a][-1]` | Finds the stage. It lists the names of all stages that start at or before paragraph `a`, and takes the **last** one, which is the stage that is currently running. | Stages starting at or before 12: I (1) and II (9). The last is "II. Sacrifice and exchange". |

Each segment is then stored as a dictionary (a labelled record) with six entries:

| Entry | Meaning |
|---|---|
| `"id"` | The segment number, 1 to 13 (`k + 1`, since `k` starts at 0) |
| `"paras"` | The paragraph range as text, for example `"12-17"` (`b - 1` is the last paragraph of the segment) |
| `"stage"` | The name of its stage |
| `"note"` | The scene description from `SEGMENT_NOTES` |
| `"text"` | The full text of the segment (the text later searched for phrases) |
| `"words"` | The number of words (`len(seg_text.split())`), needed to calculate hits per 100 words |

All 13 records are collected in the list `segments`.

### 4. Preparing three lookups

```python
seg_of = {s["id"]: s for s in segments}
stage_names = list(dict.fromkeys(s["stage"] for s in segments))
stage_words = {st: sum(s["words"] for s in segments if s["stage"] == st) for st in stage_names}
```

| Variable | What it is | How it is built | Used for |
|---|---|---|---|
| `seg_of` | A dictionary that finds a segment from its number: `seg_of[5]` returns segment 5 | Pairs each segment's `id` with the segment itself | Hover text and word counts in the charts (Step 6d) |
| `stage_names` | The four stage names, in story order, each once | `dict.fromkeys(...)` makes a dictionary whose keys are the stage names. Dictionary keys cannot repeat, and they keep the order of first appearance, so repeats are dropped and the order is preserved. `list(...)` turns the keys into a list. | The order of the bars and table columns |
| `stage_words` | The total words in each stage | For each stage name, adds up the `words` of all segments in that stage | Calculating hits per 100 words by stage |

**The stage totals are:**

| Stage | Words |
|---|---|
| I. Scarcity and shame | 495 |
| II. Sacrifice and exchange | 584 |
| III. Confrontation | 555 |
| IV. Revelation and revaluation | 423 |
| **Total** | **2,057** |

The total equals the word count of the whole story from Step 3, which confirms that no paragraph was lost or counted twice.

### 5. Showing the table

```python
pd.DataFrame([{k: v for k, v in s.items() if k != "text"} for s in segments])
```

- For each segment, the inner part builds a copy of its record **without** the `"text"` entry (`if k != "text"`), because the full text would make the table too wide to read.
- `pd.DataFrame([...])` turns that list of records into a table with the columns `id`, `paras`, `stage`, `note` and `words`.
- It is the last line of the cell, so Colab displays the table below it.

The table is the quickest way to check the division before continuing: the paragraph ranges should run from 1 to 45 without gaps, and the word counts should add up to 2,057.

### Summary

| Stage | What happens |
|---|---|
| Decide | The researcher sets where the 13 segments and 4 stages begin and describes each scene |
| Check | Assertions catch typing mistakes in the settings |
| Build | Each segment becomes a record with its text, length, stage and note |
| Prepare | Lookups give quick access to any segment and to the word total of each stage |
| Show | A table lists all segments for checking |

**How to change the division.** Only the three settings at the top need to be edited: `SEGMENT_STARTS`, `STAGES` and `SEGMENT_NOTES`. Everything below adjusts automatically, and so do the counts and charts of the next cells. If the settings are changed, the numbers quoted in the Step 7 interpretation no longer match, and the consistency check in Step 6c will show a warning.

**Link to the other steps.** The segment texts are searched for the verified phrases in Step 6b. The word counts turn raw counts into hits per 100 words in Steps 6c and 6d. The stages organise the comparison in Figure 3 and the argument of Step 7, where the original claim (shame and selling in the early stages, wisdom and narrator comments in the last) is tested against exactly this division.

In [ ]:
# paragraph number where each segment (scene) BEGINS
SEGMENT_STARTS = [1, 3, 6, 9, 12, 18, 20, 23, 27, 34, 37, 40, 45]
# paragraph number where each stage BEGINS (it must also be the start of a segment)
STAGES = [(1, "I. Scarcity and shame"), (9, "II. Sacrifice and exchange"),
          (23, "III. Confrontation"), (37, "IV. Revelation and revaluation")]
SEGMENT_NOTES = ["Della counts $1.87 and cries", "The flat and the Youngs' income",
                 "Della plans the present; the pier glass", "The two treasures; the hair falls; Della leaves",
                 "Madame Sofronie buys the hair", "Shopping; the platinum chain",
                 "Repairs at home; her reflection", "Waiting; Jim arrives",
                 "Della tells Jim she sold her hair", "The narrator's aside; the package",
                 "The combs are opened", "The watch; Jim's confession", "The narrator's coda: the magi"]

assert SEGMENT_STARTS[0] == 1 and SEGMENT_STARTS == sorted(set(SEGMENT_STARTS))
assert STAGES[0][0] == 1 and all(s in SEGMENT_STARTS for s, _ in STAGES)
assert SEGMENT_STARTS[-1] <= len(paras) and len(SEGMENT_NOTES) == len(SEGMENT_STARTS)

bounds = SEGMENT_STARTS + [len(paras) + 1]
segments = []
for k in range(len(SEGMENT_STARTS)):
    a, b = bounds[k], bounds[k + 1]
    seg_text = " ".join(paras[a - 1:b - 1])
    stage = [name for s, name in STAGES if s <= a][-1]
    segments.append({"id": k + 1, "paras": f"{a}-{b - 1}", "stage": stage, "note": SEGMENT_NOTES[k],
                     "text": seg_text, "words": len(seg_text.split())})

seg_of = {s["id"]: s for s in segments}
stage_names = list(dict.fromkeys(s["stage"] for s in segments))
stage_words = {st: sum(s["words"] for s in segments if s["stage"] == st) for st in stage_names}
pd.DataFrame([{k: v for k, v in s.items() if k != "text"} for s in segments])

,id,paras,stage,note,words
0,1,1-2,I. Scarcity and shame,Della counts $1.87 and cries,105
1,2,3-5,I. Scarcity and shame,The flat and the Youngs' income,167
2,3,6-8,I. Scarcity and shame,Della plans the present; the pier glass,223
3,4,9-11,II. Sacrifice and exchange,The two treasures; the hair falls; Della leaves,204
4,5,12-17,II. Sacrifice and exchange,Madame Sofronie buys the hair,80
5,6,18-19,II. Sacrifice and exchange,Shopping; the platinum chain,179
6,7,20-22,II. Sacrifice and exchange,Repairs at home; her reflection,121
7,8,23-26,III. Confrontation,Waiting; Jim arrives,210
8,9,27-33,III. Confrontation,Della tells Jim she sold her hair,207
9,10,34-36,III. Confrontation,The narrator's aside; the package,138


### Step 6b — Counting the phrases in each segment

## Code explanation: Step 6b, counting the phrases in each segment

**What this cell does in one sentence.** It loads the verified phrases, searches for each one in each of the 13 segments (longest phrases first, never counting the same stretch of text twice), and records every match in a table with its segment, stage, dimension and sub-type.

**Why the counting is done by code.** The language model was asked only to *find* the phrases, not to count them. Counting by code is exact and repeatable. The same phrases and the same story always give the same numbers.

**Requirements from earlier cells.** The cell uses `norm`, `word_re` and `STOPLIST` (Step 5b), `segments` (Step 6a), `text` and `pd` (Step 3), and `json` (Setup). It also needs the file `extraction/Theme_JSON.py` created in Step 5b. Those cells have to be run first.

### 1. Loading the verified phrases

```python
ns = {}
exec(open("extraction/Theme_JSON.py", encoding="utf-8").read(), ns)
themes = ns["themes"]
```

- `Theme_JSON.py` is a small Python file that contains one dictionary called `themes`.
- `exec(...)` runs the code inside that file. `ns = {}` is an empty dictionary that serves as a separate workspace, so the variables created by the file (here `themes`) are stored in `ns` and do not mix with the notebook's own variables.
- `themes = ns["themes"]` takes the dictionary out of the workspace.

`themes` has 12 keys (one per sub-type), each written as `"dimension: sub-type"` (for example `"transactional_shaming: amounts_money"`), and each key holds a list of verified phrases.

*Note:* `exec` runs whatever code the file contains, so it should only be used on a file created by the notebook itself, as here.

### 2. Building the list of search entries

```python
entries = []
for key, items in themes.items():
    d, s = [x.strip() for x in key.split(":", 1)]
    entries += [(k, d, s) for k in items]
entries += [(norm(k), None, None) for k in STOPLIST]
entries.sort(key=lambda e: -len(e[0]))
```

**Splitting the key.** `key.split(":", 1)` cuts the key at the first colon into two pieces, the dimension and the sub-type. `.strip()` removes extra spaces. So `"transactional_shaming: amounts_money"` becomes `d = "transactional_shaming"` and `s = "amounts_money"`.

**Making one entry per phrase.** Each phrase becomes a triple: `(phrase, dimension, sub-type)`. For example:

```
("pennies", "transactional_shaming", "amounts_money")
```

**Adding the stoplist.** Each stoplist phrase is added as `("at present", None, None)`. The two `None` values mean "no dimension and no sub-type". These entries take part in the matching, but they are never recorded as hits (see step 3).

**Sorting by length.** `entries.sort(key=lambda e: -len(e[0]))` puts the entries in order of phrase length, **longest first**. The minus sign reverses the usual order (shortest first). This order is the key to correct counting, as the next part shows.

### 3. The matching loop

```python
rows = []
for seg in segments:
    nt = norm(seg["text"])
    taken = [False] * len(nt)
    for k, d, s in entries:
        for m in word_re(k).finditer(nt):
            if any(taken[m.start():m.end()]):
                continue
            for i in range(m.start(), m.end()):
                taken[i] = True
            if d is not None:
                rows.append({...})
```

The loop is read from the outside in:

| Level | What it does |
|---|---|
| `for seg in segments` | Takes one segment at a time (13 rounds) |
| `nt = norm(seg["text"])` | Prepares the segment text for matching: lowercase, straight quotes, single spaces. Phrases were prepared the same way, so the two can be compared fairly. |
| `taken = [False] * len(nt)` | Creates a row of switches, one for every character of the segment, all set to `False` ("not yet used"). |
| `for k, d, s in entries` | Takes one phrase at a time, longest first. |
| `for m in word_re(k).finditer(nt)` | Finds every place in the segment where the phrase occurs as a **whole word or phrase**. `m.start()` and `m.end()` give the start and end positions of each match. |
| `if any(taken[m.start():m.end()])` | Checks whether **any character of this match is already used** by an earlier (longer) match. If yes, the match is skipped (`continue`). |
| `for i in range(...): taken[i] = True` | Marks every character of an accepted match as used. |
| `if d is not None` | Records the match only if the entry is a real phrase. Stoplist matches use up their characters but are not recorded. |

**Why the "taken" switches matter.** Without them, one stretch of text would be counted several times. Two examples:

*Example 1, a long phrase and its parts.* The sentence "I sold the watch to get the money to buy your combs" contains the phrase itself, plus the shorter phrases "sold", "watch" and "combs". Because the long phrase is searched first and its characters are marked as used, the shorter phrases cannot match inside it. The sentence counts **once**, as one hit in `price_sale_exchange`, and not as four hits in several categories.

*Example 2, a false friend.* In "too nice to use just at present", the stoplist entry "at present" is matched first and marks its characters as used. The word "present" inside it can then no longer be matched as a gift word. The stoplist entry itself is not recorded.

**Why whole-word matching matters.** `word_re` requires that no letter comes right before or after the match. For example, "hair" is not matched inside "haircut", and "wise" is not matched inside "wiser".

**What is recorded for each accepted match.** One row (a dictionary) with six entries:

| Entry | Meaning |
|---|---|
| `segment` | The segment number (1–13) |
| `stage` | The stage of that segment |
| `dimension` | The dimension of the phrase |
| `sub` | The sub-type of the phrase |
| `item` | The phrase that matched |
| `pos` | The position of the match (character number) inside the segment text, used to keep the matches in order |

### 4. Making the table and saving it

```python
df = pd.DataFrame(rows)
df.to_csv("extraction/matches.csv", index=False)
json.dump(rows, open("extraction/segment_counts.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
```

- `df` is a pandas table with one row per match. All later statistics and charts are calculated from it.
- `df.to_csv(...)` saves the table as a CSV file, which opens in Excel or Google Sheets. `index=False` leaves out the automatic row numbers.
- `json.dump(...)` saves the same matches as a JSON file, the format used in the earlier version of this project (`segment_counts.json`). `indent=2` makes it readable and `ensure_ascii=False` keeps special characters.

### 5. The summary on screen

```python
print(len(df), "matches in", len(text.split()), "words =", round(100 * len(df) / len(text.split()), 1), "per 100 words\n")
print(df.groupby("dimension").size().rename("hits").to_string())
never = [k for k, d, s in entries if d is not None and not ((df["item"] == k).any())]
print("\nPhrases that never matched on their own ...:", never or "none")
```

**Line 1.** Prints the total number of matches, the number of words in the story, and the density: matches divided by words, times 100.

**Line 2.** `df.groupby("dimension").size()` counts the rows for each dimension. `.rename("hits")` names the column, and `.to_string()` prints it as plain text.

**Lines 3 and 4.** Collect and print the phrases that **never matched on their own**: real phrases (not the stoplist) that do not appear in the item column of the table. This happens when every occurrence of a short phrase lies inside a longer phrase that matched first. The list is a transparency check: it shows which entries contribute nothing to the counts. `never or "none"` prints "none" when the list is empty.

**Expected output for the stored extraction:**

```
193 matches in 2057 words = 9.4 per 100 words

corporeal_commodification       41
objects_and_feeling             72
paradoxical_value_inversion     43
transactional_shaming           37

Phrases that never matched on their own ...: none
```

(The dimensions are printed in alphabetical order. If a different extraction is used, the numbers will differ.)

### Summary

| Stage | What happens |
|---|---|
| Load | The 152 verified phrases are read from `Theme_JSON.py` |
| Prepare | Each phrase becomes an entry with its dimension and sub-type; the stoplist is added; entries are sorted longest first |
| Match | In each segment, every phrase is searched as a whole word or phrase; a stretch of text already matched cannot be matched again |
| Record | Each accepted match becomes a row with segment, stage, dimension, sub-type, phrase and position |
| Save | `matches.csv` and `segment_counts.json` |
| Report | Total, density per 100 words, hits per dimension, and phrases that never matched on their own |

**Link to the other steps.** The table `df` and the list `rows` are the data for everything that follows: the stage tables, the statistical tests and the keyword-in-context lines (Step 6c), the five figures (Step 6d), and the numbers quoted in the Step 7 interpretation. The counting rules (whole words, longest phrase first, no text counted twice, stoplist) are the ones recorded in the README created in Step 5b, so the method can be reported exactly.

In [ ]:
ns = {}
exec(open("extraction/Theme_JSON.py", encoding="utf-8").read(), ns)
themes = ns["themes"]

entries = []                                   # (phrase, dimension, sub-type); stoplist entries have dimension None
for key, items in themes.items():
    d, s = [x.strip() for x in key.split(":", 1)]
    entries += [(k, d, s) for k in items]
entries += [(norm(k), None, None) for k in STOPLIST]
entries.sort(key=lambda e: -len(e[0]))         # longest phrase first

rows = []
for seg in segments:
    nt = norm(seg["text"])
    taken = [False] * len(nt)                  # characters already matched in this segment
    for k, d, s in entries:
        for m in word_re(k).finditer(nt):
            if any(taken[m.start():m.end()]):
                continue
            for i in range(m.start(), m.end()):
                taken[i] = True
            if d is not None:
                rows.append({"segment": seg["id"], "stage": seg["stage"], "dimension": d, "sub": s,
                             "item": k, "pos": m.start()})

df = pd.DataFrame(rows)
df.to_csv("extraction/matches.csv", index=False)
json.dump(rows, open("extraction/segment_counts.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)

print(len(df), "matches in", len(text.split()), "words =", round(100 * len(df) / len(text.split()), 1), "per 100 words\n")
print(df.groupby("dimension").size().rename("hits").to_string())
never = [k for k, d, s in entries if d is not None and not ((df["item"] == k).any())]
print("\nPhrases that never matched on their own (usually because a longer phrase already covers them):", never or "none")

193 matches in 2057 words = 9.4 per 100 words

dimension
corporeal_commodification      41
objects_and_feeling            72
paradoxical_value_inversion    43
transactional_shaming          37

Phrases that never matched on their own (usually because a longer phrase already covers them): none


### Step 6c — Counts, tests and context
This cell prints the stage table (hits and hits per 100 words), the sub-type table, the two tests, and keyword-in-context lines for the words that carry the argument of Step 7.

## Code explanation: Step 6c, tables, statistical tests and keyword-in-context lines

**What this cell does in one sentence.** It turns the table of matches into counts per stage, per sub-type and per segment, converts them into hits per 100 words, tests whether the patterns are likely to be accidents, prints the sentences around the key words, and checks that the results match the numbers quoted in the Step 7 interpretation.

**Why this cell is needed.** The raw list of 193 matches cannot be read directly. This cell summarises it so that patterns become visible, and gives each pattern two kinds of support: a statistical test (is the pattern unlikely to be chance?) and the actual sentences (does the pattern make sense in the text?).

**Requirements from earlier cells.** The cell uses `DIMENSIONS` (Step 2), `pd` (Step 3), `norm` and `word_re` (Step 5b), `segments`, `stage_names` and `stage_words` (Step 6a), and `df` (Step 6b). Those cells have to be run first.

### 1. Loading the tests and preparing the lists

```python
from scipy.stats import chisquare, spearmanr

DIMS = list(DIMENSIONS)
SUBS = [(d, s) for d in DIMS for s in DIMENSIONS[d]["subs"]]
total_words = sum(stage_words.values())
```

| Item | Meaning |
|---|---|
| `chisquare`, `spearmanr` | Two statistical tests from the `scipy` library (explained in section 4) |
| `DIMS` | The list of the four dimension names, in order |
| `SUBS` | A list of 12 pairs `(dimension, sub-type)`, one for each sub-type, in order. Used for the charts in Step 6d. |
| `total_words` | The total words of the story (2,057), the sum of the four stage lengths |

### 2. Counts and density per stage

```python
stage_counts = pd.crosstab(df["dimension"], df["stage"]).reindex(index=DIMS, columns=stage_names, fill_value=0)
stage_dens = stage_counts.div(pd.Series(stage_words), axis=1) * 100
```

**`stage_counts`: how many hits in each stage.**

- `pd.crosstab(...)` is a cross-tabulation: it counts how many rows of `df` fall into each combination of dimension (rows of the new table) and stage (columns).
- `.reindex(index=DIMS, columns=stage_names, fill_value=0)` forces the rows and columns into the story's order and fills any empty cell with 0. Without it, the stages would be listed alphabetically and a dimension with no hits in a stage would be missing.

**`stage_dens`: hits per 100 words.**

- `.div(pd.Series(stage_words), axis=1)` divides each column by the word count of its stage.
- `* 100` turns the result into hits per 100 words.

**Why density is needed.** The stages are different lengths (423 to 584 words). A longer stage would collect more hits only because it is longer. Dividing by length makes the stages comparable.

**Results for the stored extraction:**

| Dimension | Stage I | Stage II | Stage III | Stage IV |
|---|---|---|---|---|
| Transactional shaming (hits) | 29 | 5 | 3 | 0 |
| Corporeal commodification | 2 | 18 | 16 | 5 |
| Objects and feeling | 8 | 22 | 11 | 31 |
| Paradoxical value inversion | 8 | 9 | 9 | 17 |

| Hits per 100 words | Stage I | Stage II | Stage III | Stage IV |
|---|---|---|---|---|
| Transactional shaming | 5.86 | 0.86 | 0.54 | 0.00 |
| Corporeal commodification | 0.40 | 3.08 | 2.88 | 1.18 |
| Objects and feeling | 1.62 | 3.77 | 1.98 | 7.33 |
| Paradoxical value inversion | 1.62 | 1.54 | 1.62 | 4.02 |

### 3. Counts per sub-type and per segment

```python
sub_counts = pd.crosstab([df["dimension"], df["sub"]], df["stage"]).reindex(columns=stage_names, fill_value=0)
seg_counts = pd.crosstab(df["dimension"], df["segment"]).reindex(index=DIMS, columns=[s["id"] for s in segments], fill_value=0)
seg_dens = seg_counts.div(pd.Series({s["id"]: s["words"] for s in segments}), axis=1) * 100
```

| Table | Rows | Columns | Used for |
|---|---|---|---|
| `sub_counts` | The 12 sub-types (grouped by dimension) | The four stages | Seeing which sub-types drive a dimension (for example, that counting and saving words appear only in Stage I) |
| `seg_counts` | The four dimensions | The 13 segments | Locating peaks, such as the sale (segment 5) and the final paragraph (segment 13) |
| `seg_dens` | The four dimensions | The 13 segments | The same, as hits per 100 words (segments differ in length from 80 to 223 words). This table is the basis of Figure 4 and of the correlation test. |

In `seg_dens`, the divisor is built with a small dictionary `{segment id: word count}`, so each column is divided by the length of its own segment.

The four tables are printed with `.to_string()`, which shows the whole table instead of shortening it, and `.round(2)` or `.round(1)`, which limit the decimals.

### 4. The two statistical tests

```python
for d in DIMS:
    obs = stage_counts.loc[d].values
    exp = obs.sum() * pd.Series(stage_words)[stage_names].values / total_words
    chi = chisquare(obs, exp)
    rho = spearmanr(seg_dens.columns, seg_dens.loc[d].values)
    print(...)
```

For each dimension, two questions are asked.

**Test 1, chi-square: is the distribution across stages different from what stage length alone would give?**

- `obs` is the **observed** hits in the four stages, for example `[29, 5, 3, 0]` for shaming.
- `exp` is the **expected** hits if the dimension were spread evenly through the text. A stage with 25% of the words would be expected to get 25% of the hits. The formula is: all hits of the dimension × stage words ÷ total words.
- `chisquare(obs, exp)` compares the two. It returns the statistic (`chi2`; the larger, the bigger the difference between observed and expected) and the **p-value**.
- **How to read the p-value:** it is the probability of getting such an uneven distribution if the hits really were spread evenly. By convention, a p-value below .05 means the pattern is unlikely to be an accident.

**Test 2, Spearman correlation: does the density rise or fall steadily through the story?**

- `spearmanr(...)` compares two lists: the segment numbers (1 to 13) and the density in each segment.
- It returns `rho`, a number between −1 and +1, and a p-value:

| rho | Meaning |
|---|---|
| Close to −1 | The density falls steadily from the start to the end |
| Close to 0 | No steady rise or fall (it may still have bursts) |
| Close to +1 | The density rises steadily from the start to the end |

**Why two tests.** They ask different questions. Chi-square detects *uneven* distribution (a peak anywhere counts). Spearman detects a *steady trend*. A dimension can be very uneven without having a trend, for example if it peaks in the middle and in the last paragraph.

**Results for the stored extraction:**

| Dimension | chi² | p (chi-square) | rho | Reading |
|---|---|---|---|---|
| Transactional shaming | 60.7 | < .001 | −0.81 (p < .001) | Uneven **and** falling steadily |
| Corporeal commodification | 13.3 | .004 | 0.13 (not significant) | Uneven (peaks at the sale and the confession), no trend |
| Objects and feeling | 26.5 | < .001 | 0.46 (not significant) | Uneven (peaks at the shopping, the combs and the ending), no steady trend |
| Paradoxical value inversion | 9.5 | .023 | 0.04 (not significant) | Uneven (one large peak in the final paragraph), no trend |

**A necessary caution.** Both tests assume that every hit is independent of the others. In a text this is not true: neighbouring words influence each other, and one sentence can contain several hits. The p-values are therefore a rough guide to which patterns are unlikely to be accidents, not formal proof. This caution is stated again in the method note of the final report.

### 5. The keyword-in-context (KWIC) function

```python
def kwic(item, width=55, limit=8):
    k = norm(item)
    shown = 0
    for seg in segments:
        nt = norm(seg["text"])
        for m in word_re(k).finditer(nt):
            if shown >= limit:
                return
            a, b = max(0, m.start() - width), min(len(nt), m.end() + width)
            print(f"  S{seg['id']:<2} ...{nt[a:b]}...")
            shown += 1
```

**Purpose.** Numbers show *where* a word appears, but not *how* it is used. KWIC ("keyword in context") prints each occurrence of a word together with the text around it, so every pattern can be checked against the actual sentences.

**How it works.**

| Part | What it does |
|---|---|
| `def kwic(item, width=55, limit=8)` | Defines a function with a word to look for (`item`), the number of characters to show on each side (`width`, 55 by default) and the maximum number of lines (`limit`, 8 by default). |
| `k = norm(item)` | Prepares the word in the same way as the story text (lowercase, straight quotes). |
| `for seg in segments` | Goes through the segments in story order, so the lines appear in story order. |
| `word_re(k).finditer(nt)` | Finds every whole-word occurrence in the segment. |
| `if shown >= limit: return` | Stops when the limit is reached, so a very frequent word does not fill the screen. |
| `a, b = max(0, m.start() - width), min(len(nt), m.end() + width)` | Works out where the excerpt begins and ends: 55 characters before the match and 55 after it. `max(0, ...)` stops it from starting before the beginning of the text and `min(len(nt), ...)` from running past the end. |
| `print(f"  S{seg['id']:<2} ...{nt[a:b]}...")` | Prints the segment number (`S9`) and the excerpt between dots. `:<2` pads the number to two characters so the columns line up. |

**Where it is used.** The loop at the end runs the function for five words that carry the argument of Step 7:

| Word | What the lines show |
|---|---|
| `sold` | Every use of "sold" (four in Della's confession in segment 9, one in Jim's explanation in segment 12). This is the evidence for the asymmetry in the sale of the hair and the sale of the watch. |
| `hair` | The vocabulary of the body as goods. |
| `wisest` | The narrator's final verdict. |
| `magi` | The final naming of the couple. |
| `count` | The only later return of counting language, in the negated "nobody could ever count my love for you". |

**Important difference from the counts.** The printed note says that KWIC lists *every raw occurrence*. The counts in Step 6b do not: a word inside a longer phrase that was matched first (for example "sold" inside "i sold the watch to get the money to buy your combs") is not counted separately there, but it still appears in the KWIC lines. The KWIC lines are therefore a reading aid, not a second count.

### 6. The consistency check

```python
EXPECTED = {"transactional_shaming": [5.86, 0.86, 0.54, 0.0], ...}
ok = len(df) == 193 and all([round(x, 2) for x in stage_dens.loc[d].values] == EXPECTED[d] for d in DIMS)
print("\nCONSISTENCY CHECK:", "PASS - ..." if ok else "WARNING - ...")
```

**Purpose.** The interpretation in Step 7 quotes specific numbers (for example "5.86 per 100 words in Stage I"). Those numbers are typed text. If the word list, the segments or the story change, the code would calculate new numbers while the text would keep the old ones. This check compares the calculated densities with the numbers quoted in the interpretation.

- `EXPECTED` holds the four densities per stage for each dimension, as quoted in Step 7.
- `len(df) == 193` checks the total number of hits.
- `all(...)` checks that, for every dimension, the four calculated densities (rounded to two decimals) equal the expected ones.
- The printed result is either **PASS** or **WARNING**.

**What to do with a warning.** The results no longer match the interpretation. The numbers and the statements built on them in Step 7 (and the one-sentence figure captions) have to be updated before the notebook is submitted. For the stored extraction and the segments in Step 6a, the expected result is `PASS`.

### Summary

| Stage | What happens |
|---|---|
| Prepare | Lists of dimensions and sub-types; total words |
| Count | Hits per stage, sub-type and segment, from the table of matches |
| Convert | Hits per 100 words, so that stages and segments of different lengths can be compared |
| Test | Chi-square (is the stage distribution uneven?) and Spearman correlation (is there a steady trend?) |
| Read in context | KWIC lines for `sold`, `hair`, `wisest`, `magi` and `count` |
| Check | The numbers are compared with those quoted in the Step 7 interpretation |

**Link to the other steps.** The tables and tests in this cell are the numerical evidence for the Step 7 interpretation: the fall of shaming, the two peaks of the corporeal vocabulary, the late rise of objects and feeling, and the concentration of the wisdom vocabulary in the last paragraph. The variables `stage_counts`, `stage_dens`, `seg_counts` and `seg_dens` are reused directly to draw the figures in Step 6d.

In [ ]:
from scipy.stats import chisquare, spearmanr

DIMS = list(DIMENSIONS)
SUBS = [(d, s) for d in DIMS for s in DIMENSIONS[d]["subs"]]
total_words = sum(stage_words.values())

stage_counts = pd.crosstab(df["dimension"], df["stage"]).reindex(index=DIMS, columns=stage_names, fill_value=0)
stage_dens = stage_counts.div(pd.Series(stage_words), axis=1) * 100
print("HITS PER STAGE");            print(stage_counts.to_string())
print("\nHITS PER 100 WORDS");        print(stage_dens.round(2).to_string())

sub_counts = pd.crosstab([df["dimension"], df["sub"]], df["stage"]).reindex(columns=stage_names, fill_value=0)
print("\nSUB-TYPES PER STAGE (hits)");  print(sub_counts.to_string())

seg_counts = pd.crosstab(df["dimension"], df["segment"]).reindex(index=DIMS, columns=[s["id"] for s in segments], fill_value=0)
seg_dens = seg_counts.div(pd.Series({s["id"]: s["words"] for s in segments}), axis=1) * 100
print("\nHITS PER SEGMENT");           print(seg_counts.to_string())
print("\nHITS PER 100 WORDS PER SEGMENT"); print(seg_dens.round(1).to_string())

print("\nTESTS (chi-square across stages vs. stage length; Spearman rho of density against segment order)")
for d in DIMS:
    obs = stage_counts.loc[d].values
    exp = obs.sum() * pd.Series(stage_words)[stage_names].values / total_words
    chi = chisquare(obs, exp)
    rho = spearmanr(seg_dens.columns, seg_dens.loc[d].values)
    print(f"{DIMENSIONS[d]['label']:<30} chi2={chi.statistic:6.1f}  p={chi.pvalue:.4f}   rho={rho.statistic:+.2f}  p={rho.pvalue:.3f}")

def kwic(item, width=55, limit=8):
    """Keyword in context: the sentence fragments around a phrase, in story order."""
    k = norm(item)
    shown = 0
    for seg in segments:
        nt = norm(seg["text"])
        for m in word_re(k).finditer(nt):
            if shown >= limit:
                return
            a, b = max(0, m.start() - width), min(len(nt), m.end() + width)
            print(f"  S{seg['id']:<2} ...{nt[a:b]}...")
            shown += 1

print("\n(KWIC lists every raw occurrence, including words that sit inside a longer phrase counted elsewhere.)")
for word in ["sold", "hair", "wisest", "magi", "count"]:
    print(f"\nKWIC: {word}")
    kwic(word)

# ---- consistency check between these results and the interpretation written in Step 7 ----
EXPECTED = {"transactional_shaming": [5.86, 0.86, 0.54, 0.0], "corporeal_commodification": [0.4, 3.08, 2.88, 1.18],
            "objects_and_feeling": [1.62, 3.77, 1.98, 7.33], "paradoxical_value_inversion": [1.62, 1.54, 1.62, 4.02]}
ok = len(df) == 193 and all([round(x, 2) for x in stage_dens.loc[d].values] == EXPECTED[d] for d in DIMS)
print("\nCONSISTENCY CHECK:", "PASS - the numbers match the interpretation text in Step 7." if ok else
      "WARNING - your results differ from the numbers quoted in Step 7. Update the text in Step 7 before submitting.")

HITS PER STAGE
stage                        I. Scarcity and shame  II. Sacrifice and exchange  III. Confrontation  IV. Revelation and revaluation
dimension                                                                                                                         
transactional_shaming                           29                           5                   3                               0
corporeal_commodification                        2                          18                  16                               5
objects_and_feeling                              8                          22                  11                              31
paradoxical_value_inversion                      8                           9                   9                              17

HITS PER 100 WORDS
stage                        I. Scarcity and shame  II. Sacrifice and exchange  III. Confrontation  IV. Revelation and revaluation
dimension                                       

### Step 6d — Visualisation



In [ ]:
import plotly.graph_objects as go
from collections import Counter, defaultdict
from html import escape

INK, OXBLOOD = "#1f3a5f", "#8c2f39"
PALETTE = ["#1f3a5f", "#8c2f39", "#4a7c59", "#b08d57"]
FONT = "Georgia, 'Times New Roman', serif"
CONFIG = {"displaylogo": False, "toImageButtonOptions": {"format": "png", "scale": 3, "filename": "figure"}}

pretty = lambda s: s.replace("_", " ").strip().capitalize()
dim_label = {d: f"{i + 1}. {DIMENSIONS[d]['label']}" for i, d in enumerate(DIMS)}
ids = [s["id"] for s in segments]
n = len(ids)
stage_span = [(st, min(s["id"] for s in segments if s["stage"] == st),
               max(s["id"] for s in segments if s["stage"] == st)) for st in stage_names]

by_seg_dim = defaultdict(Counter)
for r in rows:
    by_seg_dim[(r["segment"], r["dimension"])][r["item"]] += 1
top = lambda c: ", ".join(f"{k} ×{v}" for k, v in c.most_common(6)) or "none"

def style(fig, h=440, left=60):
    fig.update_layout(template="simple_white", height=h, margin=dict(l=left, r=20, t=70, b=50),
                      font=dict(family=FONT, size=13, color="#222"),
                      hoverlabel=dict(font_family=FONT, bgcolor="white", align="left"),
                      legend=dict(orientation="h", y=-0.2))
    return fig

def story_axis(fig):
    fig.update_xaxes(title="Segment (in story order)", dtick=1, range=[0.5, n + 0.5])
    for st, a, b in stage_span:
        if a != 1:
            fig.add_vline(x=a - 0.5, line_width=1.5, line_dash="dot", line_color=OXBLOOD)
        fig.add_annotation(x=(a + b) / 2, y=1.0, yref="paper", yanchor="bottom", showarrow=False,
                           text=st, font=dict(size=11, color=OXBLOOD))
    return fig

# Figure 1: heatmap, dimension x segment (colour = hits per 100 words, number = raw hits)
raw = [[sum(by_seg_dim[(i, d)].values()) for i in ids] for d in DIMS]
dens = [[100 * raw[a][b] / seg_of[i]["words"] for b, i in enumerate(ids)] for a in range(len(DIMS))]
hover = [[f"<b>{dim_label[d]}</b><br>Segment {i}: {seg_of[i]['note']}<br>{raw[a][b]} hit(s) = {dens[a][b]:.1f} per 100 words"
          f"<br>{top(by_seg_dim[(i, d)])}" for b, i in enumerate(ids)] for a, d in enumerate(DIMS)]
fig1 = go.Figure(go.Heatmap(z=dens, x=ids, y=[dim_label[d] for d in DIMS], customdata=hover,
                            text=[[str(v) for v in row] for row in raw], texttemplate="%{text}",
                            hovertemplate="%{customdata}<extra></extra>",
                            colorscale=[[0, "#f3eee4"], [1, INK]], xgap=3, ygap=3,
                            colorbar=dict(title="Hits per<br>100 words")))
style(fig1, 380, 200); fig1.update_yaxes(autorange="reversed"); story_axis(fig1)

# Figure 2: stages compared, hits per 100 words
fig2 = go.Figure()
for di, d in enumerate(DIMS):
    fig2.add_bar(x=stage_names, y=[stage_dens.loc[d, st] for st in stage_names], name=dim_label[d],
                 marker_color=PALETTE[di],
                 customdata=[[int(stage_counts.loc[d, st]), stage_words[st]] for st in stage_names],
                 hovertemplate="%{x}<br>" + dim_label[d] + "<br>%{y:.2f} per 100 words"
                               "<br>(%{customdata[0]} hits in %{customdata[1]} words)<extra></extra>")
style(fig2, 430); fig2.update_layout(barmode="group"); fig2.update_yaxes(title="Hits per 100 words")

# Figure 3: timeline, one line per dimension
fig3 = go.Figure()
for di, d in enumerate(DIMS):
    fig3.add_trace(go.Scatter(x=ids, y=[seg_dens.loc[d, i] for i in ids], mode="lines+markers", name=dim_label[d],
                              line=dict(color=PALETTE[di], width=2.5),
                              customdata=[[int(seg_counts.loc[d, i]), seg_of[i]["words"]] for i in ids],
                              hovertemplate="Segment %{x}<br>" + dim_label[d] + "<br>%{y:.2f} per 100 words"
                                            "<br>(%{customdata[0]} hits in %{customdata[1]} words)<extra></extra>"))
style(fig3, 440); fig3.update_yaxes(title="Hits per 100 words"); story_axis(fig3)

# Figure 4: the most frequent phrases
common = Counter(r["item"] for r in rows).most_common(15)
item_dim = {r["item"]: r["dimension"] for r in rows}
fig4 = go.Figure(go.Bar(x=[v for _, v in common][::-1], y=[k for k, _ in common][::-1], orientation="h",
                        marker_color=[PALETTE[DIMS.index(item_dim[k])] for k, _ in common][::-1],
                        hovertemplate="%{y}: %{x} hit(s)<extra></extra>"))
style(fig4, 460, 230); fig4.update_xaxes(title="Number of occurrences")
fig4.update_layout(showlegend=False)

FIGS = [fig1, fig2, fig3, fig4]
for f in FIGS:
    f.show(config=CONFIG)

# STEP 7 — Interpretation of the Result

In [ ]:
INTERPRETATION = {
    1: "Shame vocabulary is densest in segment 1 (12.4 hits per 100 words) and is absent from segment 9 onwards; corporeal vocabulary peaks at the sale (segment 5, 10.0) and again at the confession (segment 9, 6.8); objects and feeling peak when the combs are opened (segment 11, 12.0); inversion peaks only in the final paragraph (segment 13, 13.1).",
    2: "Shaming falls from 5.86 to 0.00 hits per 100 words across the four stages, corporeal commodification peaks in Stages II and III (3.08 and 2.88), and objects and feeling and inversion both peak in Stage IV (7.33 and 4.02).",
    3: "Only shaming declines steadily (Spearman rho = -0.81); the other three dimensions move in bursts tied to scenes: the sale, the confession, the opening of the combs and the final paragraph.",
    4: "\"hair\" is the most frequent single item (9), followed by \"beautiful\" (5) and \"sold\", \"combs\" and \"gifts\" (4 each): the vocabulary of value is concentrated in a few words about the body and the gifts.",
}
HOW = {
    1: "Each row is one dimension and each column is one segment of the story, in order. The number in each square is the number of hits; the shade shows hits per 100 words, so short and long segments can be compared. Dotted red lines mark where a new stage begins.",
    2: "The story is grouped into four stages. Each bar shows hits per 100 words, so a longer stage does not look stronger only because it is longer.",
    3: "Each line is one dimension followed across the segments in story order, in hits per 100 words. A rising line means that the dimension grows denser as the story moves on.",
    4: "The fifteen most frequent phrases in the extraction, coloured by dimension.",
}
TITLES = {1: "Figure 1. Where does each dimension appear in the story?",
          2: "Figure 2. How do the stages differ?",
          3: "Figure 3. How does each dimension change across the story?",
          4: "Figure 4. Which phrases carry the most weight?"}

In [ ]:
ESSAY = r"""
# Interpretation: how the narrator's language turns economic loss into emotional value in "The Gift of the Magi"

## 7.1 Basis of the interpretation

This interpretation rests on the results of Step 6: the table of 193 matched phrases, the counts per stage and segment, the two statistical tests, the keyword-in-context lines, and the four figures. It follows the reader-oriented approach of the project: the four dimensions came from the initial reading of the story, and the counts are used to check, refine and sometimes correct that first impression.

The counts are descriptive. They show where each kind of language appears in the story and how often. They do not show tone or meaning on their own, so every pattern below is read together with the passage of the story in which it occurs.

Throughout, results are given as hits per 100 words, so that stages and segments of different lengths can be compared fairly. Stage I has 495 words, Stage II 584, Stage III 555 and Stage IV 423. The segments range from 80 to 223 words.

## 7.2 The overall picture

The extraction found 193 phrase occurrences in 2,057 words, which is 9.4 hits per 100 words. Each occurrence belongs to exactly one sub-type, so nothing is counted twice. By dimension the totals are:

- Objects and feeling: 72 hits (37% of all hits)
- Paradoxical value inversion: 43 hits
- Corporeal commodification: 41 hits
- Transactional shaming: 37 hits

The vocabulary of objects and feeling is therefore the largest single field in the story. The other three dimensions are of similar size, but, as the figures show, they are located in very different places.

## 7.3 Figure 1: where each dimension appears (heatmap)

The heatmap shows each dimension in each of the 13 segments. The number in each square is the count of hits, and the shade shows hits per 100 words.

**Transactional shaming (row 1)** is dark only at the start. Segment 1 (Della counts her money) has 13 hits in 105 words, which is 12.4 per 100 words. Segment 2 (the flat and the income) has 5 hits and segment 3 (Della plans the present) has 11 hits. After that the row fades: 1, 0, 3, 1 and 3 hits in segments 4 to 8, and then **zero in every segment from 9 to 13**. Shame and money language belongs to the opening of the story and then disappears.

**Corporeal commodification (row 2)** has two clear peaks:
- Segment 5, the sale to Madame Sofronie, has 8 hits in only 80 words (10.0 per 100 words). Every hit in this segment belongs to this one dimension, and there are no hits at all for shame, objects or inversion. The scene is entirely about the transaction.
- Segment 9, where Della tells Jim what she has done, has 14 hits (6.8 per 100 words), the highest count of any segment for this dimension.

Between and around the peaks, the row is almost empty: 0 hits in segments 1 and 2, and 0 in segments 6 and 10.

**Objects and feeling (row 3)** is the darkest row overall and has two main peaks: segment 6, the search for the platinum chain (14 hits, 7.8 per 100 words), and segment 11, the opening of the combs (18 hits in 150 words, 12.0 per 100 words, the highest density of any square in this row). Smaller concentrations occur in segment 4 (7 hits, the two treasures), segment 9 (8 hits), segment 12 (8 hits, Jim's confession about the watch) and segment 13 (5 hits). The row is empty in segment 5 (the sale) and segment 10.

**Paradoxical value inversion (row 4)** is light in almost every square and then very dark at the end: segment 13, the narrator's closing paragraph, has 16 hits in 122 words, which is 13.1 per 100 words. The next highest squares are segment 10 (6 hits, 4.3 per 100 words) and segment 2 (5 hits, 3.0 per 100 words).

**The busiest segment.** The final paragraph (segment 13) has 22 hits in 122 words, which is 18.0 per 100 words, the highest overall density of any segment. Segment 11 (14.7 per 100 words) and segment 1 (14.3 per 100 words) follow. The story is therefore most densely marked at the start (money), at the opening of the combs (objects and feeling) and at the end (the narrator's verdict).

## 7.4 Figure 2: how the four stages differ (grouped bars)

The stage chart combines the segments into four stages and shows hits per 100 words.

| Dimension | Stage I | Stage II | Stage III | Stage IV |
|---|---|---|---|---|
| Transactional shaming | 5.86 | 0.86 | 0.54 | 0.00 |
| Corporeal commodification | 0.40 | 3.08 | 2.88 | 1.18 |
| Objects and feeling | 1.62 | 3.77 | 1.98 | 7.33 |
| Paradoxical value inversion | 1.62 | 1.54 | 1.62 | 4.02 |

**Shaming falls in every stage.** It drops from 5.86 in Stage I to 0.86, 0.54 and finally 0.00. Stage I alone holds 29 of its 37 hits.

**Corporeal commodification rises and then falls.** It is almost absent in Stage I (0.40), reaches its highest level in Stage II (3.08), stays nearly as high in Stage III (2.88) and falls to 1.18 in Stage IV. The sale (Stage II) and the confession (Stage III) are the two high stages.

**Objects and feeling is the strongest in Stage IV**, at 7.33, which is almost twice the next highest stage (3.77 in Stage II). In Stage III it drops to 1.98, so the language of objects is not a steady rise. It is strongest during the shopping and in the final scenes.

**Value inversion is flat and then jumps.** It stays between 1.5 and 1.6 for Stages I to III and rises to 4.02 in Stage IV. Within Stage IV, 16 of the 17 hits come from the last paragraph.

**Statistical check.** A chi-square test compares the actual distribution across stages with an even spread (the spread expected from stage length alone, 3 degrees of freedom). All four dimensions differ clearly from an even spread: shaming (chi-square 60.7, p < .001), corporeal commodification (13.3, p = .004), objects and feeling (26.5, p < .001) and inversion (9.5, p = .023). None of the four patterns is likely to be an accident of where the words happen to fall. Because neighbouring words influence each other, the p-values are a rough guide and not formal proof.

## 7.5 Figure 3: change across the story (timeline)

The timeline follows each dimension through the 13 segments in order.

**Shaming is the only dimension with a steady trend.** Its line starts high and falls. The correlation between segment order and density is strongly negative (rho = -0.81, p < .001).

**The other three dimensions move in bursts, not trends.** Their correlations are small and not significant (corporeal 0.13, objects and feeling 0.46, inversion 0.04). Each line rises sharply at particular scenes and drops in between:
- Corporeal commodification spikes at the sale (segment 5) and the confession (segment 9).
- Objects and feeling spikes at the chain (segment 6) and the combs (segment 11). Between them, in segment 7 (Della's repairs at home), it falls to 1 hit (0.8 per 100 words), so the language of objects rises and falls with the scenes in which gifts are handled.
- Inversion stays low until it rises abruptly in the last point of the line.

The timeline therefore shows that only the language of shame changes gradually. The other three change scene by scene.

## 7.6 Figure 4: the most frequent phrases

The ranking of the fifteen most frequent phrases shows which words carry the weight of the analysis.

- **"hair" is the most frequent single item, with 9 occurrences.** It appears once in segment 3 (Della's plan), three times in segment 4 (the hair falls), four times in segment 9 (the confession) and once in segment 11 (the combs). The word is mentioned before, during and after the sale.
- **"beautiful" follows with 5 occurrences**, and **"sold", "combs" and "gifts"** have 4 each. All four occurrences of "sold" are in segment 9, in Della's confession. Jim's own sale is stated once, as a complete sentence ("I sold the watch to get the money to buy your combs") in segment 12.
- The ranking is dominated by words about the body and about the gifts. Words of money and counting appear in the list less often, because they were spread over many different phrases ("pennies", "$8", "saved", "counted") and each of them occurs only a few times.

## 7.7 Interpretation by dimension

### Transactional shaming: how counting becomes shame

Within this dimension, the three sub-types are not equal. Words for amounts of money have 16 hits (13 in Stage I and 3 in Stage II), counting, saving and bargaining has 11 hits (all 11 in Stage I) and words of shame and want have 10 hits (5, 2 and 3 in Stages I to III).

The opening paragraph shows the pattern. The sum "one dollar and eighty-seven cents" is stated twice in 67 words, "pennies" appears twice, and the narrator describes how the pennies were saved "by bulldozing the grocer and the vegetable man and the butcher" until "one's cheeks burned with the silent imputation of parsimony". The shame here is not caused by poverty alone. It is caused by being watched while bargaining: the grocer, the vegetable man and the butcher are the observers.

The vocabulary of shame and want continues into the description of the home (segment 2, "mendicancy squad"), and returns in smaller form when Jim "sometimes looked at it on the sly" at his watch because of its old strap, and when the narrator mentions that he "needed a new overcoat and he was without gloves". These are all marks of lack that are felt as embarrassment.

Counting vocabulary is completely absent after Stage I. The only later mention of counting is a negation, in Della's words to Jim: "nobody could ever count my love for you". The narrator's remark in segment 10, "Eight dollars a week or a million a year—what is the difference?", also removes the importance of quantity. The story starts in numbers and moves away from them, which is the first step in turning economic loss into emotional value.

### Corporeal commodification: the sale of the hair

This dimension is mainly about hair. The three sub-types differ in timing:
- **Hair as goods (17 hits)** is spread across Stages II and III (6 and 8 hits) and is the main content of the dimension.
- **Price, sale and exchange (9 hits)** is concentrated in Stage III (4 hits) and Stage II (3 hits), with the remainder in Stage IV.
- **Reactions to the sale (15 hits)** are mostly in Stage II (9 hits, 1.54 per 100 words): "panting", "a tear or two", "faltered", "nervously", "truant schoolboy". They describe Della's body and her appearance after the sale.

The sale scene (segment 5) is the most concentrated passage in the story. In 80 words it contains the sign "Hair Goods of All Kinds", the question "Will you buy my hair?", Madame's "I buy hair" and "lifting the mass with a practised hand", the price "Twenty dollars", and Della's "Give it to me quick". The hair is turned from a part of Della into goods with a price within a few lines.

The second peak (segment 9) is a different use of the same vocabulary. Della uses the words of sale as a confession and an appeal to Jim: "I had my hair cut off and sold", "Cut it off and sold it", "It's sold, I tell you—sold and gone, too". This is why the corporeal dimension remains high in Stage III (2.88 per 100 words) and why the original expectation that selling language belongs to the first two stages is only partly correct.

**An unequal distribution of the language of sacrifice.** The data show that the two sacrifices are not described in the same way. Della's sacrifice takes up two segments with high density (segments 5 and 9), nine reaction words and all four occurrences of "sold". Jim's sacrifice, the sale of the watch, is reported in one sentence in segment 12, and there are no words of bodily reaction attached to it. The watch is also described as an inherited object, "his father's and his grandfather's". The language of the story spends far more space on the sale of the body than on the sale of the object. This is an observation about the distribution of words, not a statement of what the story intends: the last paragraph calls both gifts "the greatest treasures of their house".

### Objects and feeling: where value is attached

This is the largest dimension, with 72 hits (3.50 per 100 words). Its three sub-types are gifts and possessions (30 hits), feelings attached (22 hits) and qualities of objects (20 hits).

**Feelings grow steadily.** Hits for feelings per 100 words are 0.61, 0.68, 0.90 and 2.36 in Stages I to IV. This is the only sub-type among the four dimensions that rises in every stage.

**Gifts and possessions** are absent in Stage I (0 hits), appear in Stage II (2.05 per 100 words), fall in Stage III (0.54) and reach their highest level in Stage IV (3.55).

**Segment 6, the chain.** In this passage (14 hits) the platinum fob chain is described with qualities that Della also attributes to Jim: "simple and chaste in design", "Quietness and value—the description applied to both". The value of the chain is explained by its likeness to Jim, so the object is valued for the person.

**Segment 11, the combs.** This segment has the highest density of this dimension (12.0 per 100 words) and includes 8 words of feeling: "worshipped", "craved", "yearned", "ecstatic scream of joy", "hysterical tears and wails". The combs are described with the language of beauty and expense ("beautiful", "pure tortoise shell", "jewelled rims", "expensive", "coveted adornments") at the exact moment when they cannot be used, because the hair they were bought for has been sold.

This is the main finding of the dimension: the strongest language of value and desire is attached to the combs when they have lost their practical use. The loss does not reduce the value of the gift in the language of the story. It increases it.

### Paradoxical value inversion: the narrator's voice

The three sub-types behave differently.

**Narrator comments (15 hits)** are not concentrated at the end. Stage I has 6 hits (1.21 per 100 words), the other stages 3 each. The final paragraph has 3 hits in 122 words (about 2.5 per 100 words), which is the highest density of any single segment. The narrator is therefore present at the start and at the end of the story, with a lower level in between. The comments also have different functions. Early comments, such as "Which is all very good" and the phrase "did not exactly beggar description", appear in the same segment as the shame vocabulary (segment 2) and make the poverty lighter in tone. The comments in the middle of the story ("Forget the hashed metaphor", "dear friends") speak directly to the reader. The comments at the end give the narrator's final verdict.

**Wisdom and foolishness (14 hits)** are almost entirely at the end: 11 of the 14 hits are in segment 13 ("wise men", "wonderfully wise men", "foolish children", "unwisely", "wisest", "magi"). The word "wise" does not occur before the last paragraph. Before it, the vocabulary of reason is used for moments of calculation and blankness: "prudence and reason" when Della calms down after shopping (segment 7), "an air almost of idiocy" for Jim after hearing about the hair (segment 9) and "A mathematician or a wit would give you the wrong answer" in the narrator's aside (segment 10). The reader is therefore first shown that calculation is not the right way to judge what has happened, and only in the last paragraph is told who is wise.

**Ironic register (14 hits)** is thin in Stage I (0.40 per 100 words) and stronger in Stages II to IV (0.86, 0.72 and 0.71). An example is segment 4, where the narrator compares Della's hair and Jim's watch with the jewels of the queen of Sheba and the treasures of King Solomon. The hair and the watch are raised above royal treasure through exaggeration, and this prepares the valuation stated openly at the end.

## 7.8 Testing the original claim

The original claim was: *words of shame, counting and selling are concentrated in the first two stages, while words of wisdom and the narrator's comments are concentrated in the last stage.*

| Part of the claim | Result | Evidence |
|---|---|---|
| Shame and counting words are early | **Supported, more strongly than claimed** | 29 of 37 shame hits and all 11 counting hits are in Stage I alone; shame falls steadily (rho = -0.81) |
| Selling words are in the first two stages | **Partly supported** | Selling and body vocabulary is highest in Stages II and III (3.08 and 2.88 per 100 words) and returns in Della's confession (segment 9) |
| Wisdom words are in the last stage | **Supported, with a qualification** | 11 of 14 hits are in a single paragraph (segment 13) |
| Narrator comments are in the last stage | **Not supported as stated** | They are highest in Stage I (6 hits) and again in the final paragraph (3 hits); the narrator is present at both ends |

## 7.9 Answer to the research question and return to the initial impression

The research question was: *How does the narrator's language turn economic loss into emotional value, and what does the reader learn to call "wise"?*

The initial impression, formed after the partial reading, was that money and love pull against each other and that the narrator's language turns loss into value. The results support this impression and show how it works. The conversion happens in four overlapping moves:

1. **Counting and shame come first.** The story opens in money (Stage I: 5.86 shame hits per 100 words) and in the embarrassment of scarcity.
2. **Numbers disappear.** Counting words are absent after Stage I and amounts of money are absent after Stage II.
3. **The sale of the hair is told in the language of goods and then in the language of confession.** The body is priced in segment 5 and the sale is spoken of again in segment 9.
4. **Feeling and value move onto the gifts.** The words of beauty, desire and joy are strongest around the combs (segment 11) and in the final stage (7.33 per 100 words), at the moment the gifts cannot be used.

**What the reader learns to call "wise".** The narrator names the result only in the last paragraph, where 11 of the 14 wisdom words appear. The reader is led to feel the value of the gifts before being told to call it wisdom, and the earlier comments about calculation prepare the final reversal: the two characters who gave up their most valuable possessions are called "the wisest".

The results also correct the first impression in two ways. The conversion is not one smooth process but several, running at different speeds. And the sale of the body is as prominent in the language as money, and returns in Stage III after the money language has gone.

## 7.10 Position of the reader and role of the model

The story was read by a college student from a busy, multilingual urban Indian setting, with English as an academic language rather than the language of the home. This position shaped what was noticed first: money, family obligation and feeling. Several of the most telling shame words, such as "parsimony", "mendicancy" and "meretricious", belong to a formal literary register that is easy to pass over when reading for plot and feeling. The extraction step made them visible, so the model complemented the reading rather than replacing it. The choice of the four dimensions, however, came from the reading itself. Another reader might begin from different concerns and arrive at a different set of dimensions.

## 7.11 Limitations

- **The method depends on a word list.** Words such as "present", "nice" and "hair" can have several senses, and counting treats all senses alike. One false positive ("just at present", meaning "now", not "gift") was caught with a stoplist, but others may remain.
- **The extraction was done by a language model** from a prompt written for this study. Every phrase was checked against the text, but the choice of which phrases to extract could differ for another model or another reader.
- **The division into 13 segments and 4 stages** was decided by the researcher from changes of scene. A different division would change the stage figures, although the strongest patterns (the fall of shaming and the final peak of inversion) are also visible in the individual segments.
- **The story is short.** A short segment, such as segment 5 with 80 words, can show a high density from a few hits.
- **Counts show vocabulary, not tone.** Irony in particular is only partly captured by phrases.
- **The statistical tests are a guide only**, because hits within a passage are not independent.

## 7.12 Conclusion

**Answer to the research question.** The narrator's language turns economic loss into emotional value gradually, in four steps, and teaches the reader to call the two characters "wise" only at the very end.

1. **Counting and shame come first.** The story opens in money: shame vocabulary is at 5.86 hits per 100 words in Stage I and falls steadily to 0.00 by Stage IV.
2. **The numbers disappear.** Counting words are absent after Stage I.
3. **The hair is sold in the language of goods and then of confession.** The sale (segment 5) and Della's confession (segment 9) are the two peaks of the corporeal vocabulary.
4. **Feeling and value move onto the gifts.** Words of beauty, desire and joy are strongest around the combs (segment 11) and in Stage IV (7.33 per 100 words), at the moment the gifts cannot be used.

**What the reader learns to call "wise".** The narrator names the result in the last paragraph, where 11 of the 14 wisdom words appear: the couple who sacrificed their most valuable possessions for each other are "the wisest" and "the magi". Earlier comments about calculation ("A mathematician or a wit would give you the wrong answer") prepare this reversal.

**Two additional findings.** The original claim is only partly supported: shame and counting are early, as expected, but selling language returns in Stage III and the narrator's comments appear at both the start and the end. The language of sacrifice is also unequal: most of the vocabulary of selling and of bodily reaction belongs to Della.
"""
ESSAY = ESSAY.strip()
open("docs/interpretation.md", "w", encoding="utf-8").write(ESSAY)
display(Markdown(ESSAY))

# Interpretation: how the narrator's language turns economic loss into emotional value in "The Gift of the Magi"

## 7.1 Basis of the interpretation

This interpretation rests on the results of Step 6: the table of 193 matched phrases, the counts per stage and segment, the two statistical tests, the keyword-in-context lines, and the four figures. It follows the reader-oriented approach of the project: the four dimensions came from the initial reading of the story, and the counts are used to check, refine and sometimes correct that first impression.

The counts are descriptive. They show where each kind of language appears in the story and how often. They do not show tone or meaning on their own, so every pattern below is read together with the passage of the story in which it occurs.

Throughout, results are given as hits per 100 words, so that stages and segments of different lengths can be compared fairly. Stage I has 495 words, Stage II 584, Stage III 555 and Stage IV 423. The segments range from 80 to 223 words.

## 7.2 The overall picture

The extraction found 193 phrase occurrences in 2,057 words, which is 9.4 hits per 100 words. Each occurrence belongs to exactly one sub-type, so nothing is counted twice. By dimension the totals are:

- Objects and feeling: 72 hits (37% of all hits)
- Paradoxical value inversion: 43 hits
- Corporeal commodification: 41 hits
- Transactional shaming: 37 hits

The vocabulary of objects and feeling is therefore the largest single field in the story. The other three dimensions are of similar size, but, as the figures show, they are located in very different places.

## 7.3 Figure 1: where each dimension appears (heatmap)

The heatmap shows each dimension in each of the 13 segments. The number in each square is the count of hits, and the shade shows hits per 100 words.

**Transactional shaming (row 1)** is dark only at the start. Segment 1 (Della counts her money) has 13 hits in 105 words, which is 12.4 per 100 words. Segment 2 (the flat and the income) has 5 hits and segment 3 (Della plans the present) has 11 hits. After that the row fades: 1, 0, 3, 1 and 3 hits in segments 4 to 8, and then **zero in every segment from 9 to 13**. Shame and money language belongs to the opening of the story and then disappears.

**Corporeal commodification (row 2)** has two clear peaks:
- Segment 5, the sale to Madame Sofronie, has 8 hits in only 80 words (10.0 per 100 words). Every hit in this segment belongs to this one dimension, and there are no hits at all for shame, objects or inversion. The scene is entirely about the transaction.
- Segment 9, where Della tells Jim what she has done, has 14 hits (6.8 per 100 words), the highest count of any segment for this dimension.

Between and around the peaks, the row is almost empty: 0 hits in segments 1 and 2, and 0 in segments 6 and 10.

**Objects and feeling (row 3)** is the darkest row overall and has two main peaks: segment 6, the search for the platinum chain (14 hits, 7.8 per 100 words), and segment 11, the opening of the combs (18 hits in 150 words, 12.0 per 100 words, the highest density of any square in this row). Smaller concentrations occur in segment 4 (7 hits, the two treasures), segment 9 (8 hits), segment 12 (8 hits, Jim's confession about the watch) and segment 13 (5 hits). The row is empty in segment 5 (the sale) and segment 10.

**Paradoxical value inversion (row 4)** is light in almost every square and then very dark at the end: segment 13, the narrator's closing paragraph, has 16 hits in 122 words, which is 13.1 per 100 words. The next highest squares are segment 10 (6 hits, 4.3 per 100 words) and segment 2 (5 hits, 3.0 per 100 words).

**The busiest segment.** The final paragraph (segment 13) has 22 hits in 122 words, which is 18.0 per 100 words, the highest overall density of any segment. Segment 11 (14.7 per 100 words) and segment 1 (14.3 per 100 words) follow. The story is therefore most densely marked at the start (money), at the opening of the combs (objects and feeling) and at the end (the narrator's verdict).

## 7.4 Figure 2: how the four stages differ (grouped bars)

The stage chart combines the segments into four stages and shows hits per 100 words.

| Dimension | Stage I | Stage II | Stage III | Stage IV |
|---|---|---|---|---|
| Transactional shaming | 5.86 | 0.86 | 0.54 | 0.00 |
| Corporeal commodification | 0.40 | 3.08 | 2.88 | 1.18 |
| Objects and feeling | 1.62 | 3.77 | 1.98 | 7.33 |
| Paradoxical value inversion | 1.62 | 1.54 | 1.62 | 4.02 |

**Shaming falls in every stage.** It drops from 5.86 in Stage I to 0.86, 0.54 and finally 0.00. Stage I alone holds 29 of its 37 hits.

**Corporeal commodification rises and then falls.** It is almost absent in Stage I (0.40), reaches its highest level in Stage II (3.08), stays nearly as high in Stage III (2.88) and falls to 1.18 in Stage IV. The sale (Stage II) and the confession (Stage III) are the two high stages.

**Objects and feeling is the strongest in Stage IV**, at 7.33, which is almost twice the next highest stage (3.77 in Stage II). In Stage III it drops to 1.98, so the language of objects is not a steady rise. It is strongest during the shopping and in the final scenes.

**Value inversion is flat and then jumps.** It stays between 1.5 and 1.6 for Stages I to III and rises to 4.02 in Stage IV. Within Stage IV, 16 of the 17 hits come from the last paragraph.

**Statistical check.** A chi-square test compares the actual distribution across stages with an even spread (the spread expected from stage length alone, 3 degrees of freedom). All four dimensions differ clearly from an even spread: shaming (chi-square 60.7, p < .001), corporeal commodification (13.3, p = .004), objects and feeling (26.5, p < .001) and inversion (9.5, p = .023). None of the four patterns is likely to be an accident of where the words happen to fall. Because neighbouring words influence each other, the p-values are a rough guide and not formal proof.

## 7.5 Figure 3: change across the story (timeline)

The timeline follows each dimension through the 13 segments in order.

**Shaming is the only dimension with a steady trend.** Its line starts high and falls. The correlation between segment order and density is strongly negative (rho = -0.81, p < .001).

**The other three dimensions move in bursts, not trends.** Their correlations are small and not significant (corporeal 0.13, objects and feeling 0.46, inversion 0.04). Each line rises sharply at particular scenes and drops in between:
- Corporeal commodification spikes at the sale (segment 5) and the confession (segment 9).
- Objects and feeling spikes at the chain (segment 6) and the combs (segment 11). Between them, in segment 7 (Della's repairs at home), it falls to 1 hit (0.8 per 100 words), so the language of objects rises and falls with the scenes in which gifts are handled.
- Inversion stays low until it rises abruptly in the last point of the line.

The timeline therefore shows that only the language of shame changes gradually. The other three change scene by scene.

## 7.6 Figure 4: the most frequent phrases

The ranking of the fifteen most frequent phrases shows which words carry the weight of the analysis.

- **"hair" is the most frequent single item, with 9 occurrences.** It appears once in segment 3 (Della's plan), three times in segment 4 (the hair falls), four times in segment 9 (the confession) and once in segment 11 (the combs). The word is mentioned before, during and after the sale.
- **"beautiful" follows with 5 occurrences**, and **"sold", "combs" and "gifts"** have 4 each. All four occurrences of "sold" are in segment 9, in Della's confession. Jim's own sale is stated once, as a complete sentence ("I sold the watch to get the money to buy your combs") in segment 12.
- The ranking is dominated by words about the body and about the gifts. Words of money and counting appear in the list less often, because they were spread over many different phrases ("pennies", "$8", "saved", "counted") and each of them occurs only a few times.

## 7.7 Interpretation by dimension

### Transactional shaming: how counting becomes shame

Within this dimension, the three sub-types are not equal. Words for amounts of money have 16 hits (13 in Stage I and 3 in Stage II), counting, saving and bargaining has 11 hits (all 11 in Stage I) and words of shame and want have 10 hits (5, 2 and 3 in Stages I to III).

The opening paragraph shows the pattern. The sum "one dollar and eighty-seven cents" is stated twice in 67 words, "pennies" appears twice, and the narrator describes how the pennies were saved "by bulldozing the grocer and the vegetable man and the butcher" until "one's cheeks burned with the silent imputation of parsimony". The shame here is not caused by poverty alone. It is caused by being watched while bargaining: the grocer, the vegetable man and the butcher are the observers.

The vocabulary of shame and want continues into the description of the home (segment 2, "mendicancy squad"), and returns in smaller form when Jim "sometimes looked at it on the sly" at his watch because of its old strap, and when the narrator mentions that he "needed a new overcoat and he was without gloves". These are all marks of lack that are felt as embarrassment.

Counting vocabulary is completely absent after Stage I. The only later mention of counting is a negation, in Della's words to Jim: "nobody could ever count my love for you". The narrator's remark in segment 10, "Eight dollars a week or a million a year—what is the difference?", also removes the importance of quantity. The story starts in numbers and moves away from them, which is the first step in turning economic loss into emotional value.

### Corporeal commodification: the sale of the hair

This dimension is mainly about hair. The three sub-types differ in timing:
- **Hair as goods (17 hits)** is spread across Stages II and III (6 and 8 hits) and is the main content of the dimension.
- **Price, sale and exchange (9 hits)** is concentrated in Stage III (4 hits) and Stage II (3 hits), with the remainder in Stage IV.
- **Reactions to the sale (15 hits)** are mostly in Stage II (9 hits, 1.54 per 100 words): "panting", "a tear or two", "faltered", "nervously", "truant schoolboy". They describe Della's body and her appearance after the sale.

The sale scene (segment 5) is the most concentrated passage in the story. In 80 words it contains the sign "Hair Goods of All Kinds", the question "Will you buy my hair?", Madame's "I buy hair" and "lifting the mass with a practised hand", the price "Twenty dollars", and Della's "Give it to me quick". The hair is turned from a part of Della into goods with a price within a few lines.

The second peak (segment 9) is a different use of the same vocabulary. Della uses the words of sale as a confession and an appeal to Jim: "I had my hair cut off and sold", "Cut it off and sold it", "It's sold, I tell you—sold and gone, too". This is why the corporeal dimension remains high in Stage III (2.88 per 100 words) and why the original expectation that selling language belongs to the first two stages is only partly correct.

**An unequal distribution of the language of sacrifice.** The data show that the two sacrifices are not described in the same way. Della's sacrifice takes up two segments with high density (segments 5 and 9), nine reaction words and all four occurrences of "sold". Jim's sacrifice, the sale of the watch, is reported in one sentence in segment 12, and there are no words of bodily reaction attached to it. The watch is also described as an inherited object, "his father's and his grandfather's". The language of the story spends far more space on the sale of the body than on the sale of the object. This is an observation about the distribution of words, not a statement of what the story intends: the last paragraph calls both gifts "the greatest treasures of their house".

### Objects and feeling: where value is attached

This is the largest dimension, with 72 hits (3.50 per 100 words). Its three sub-types are gifts and possessions (30 hits), feelings attached (22 hits) and qualities of objects (20 hits).

**Feelings grow steadily.** Hits for feelings per 100 words are 0.61, 0.68, 0.90 and 2.36 in Stages I to IV. This is the only sub-type among the four dimensions that rises in every stage.

**Gifts and possessions** are absent in Stage I (0 hits), appear in Stage II (2.05 per 100 words), fall in Stage III (0.54) and reach their highest level in Stage IV (3.55).

**Segment 6, the chain.** In this passage (14 hits) the platinum fob chain is described with qualities that Della also attributes to Jim: "simple and chaste in design", "Quietness and value—the description applied to both". The value of the chain is explained by its likeness to Jim, so the object is valued for the person.

**Segment 11, the combs.** This segment has the highest density of this dimension (12.0 per 100 words) and includes 8 words of feeling: "worshipped", "craved", "yearned", "ecstatic scream of joy", "hysterical tears and wails". The combs are described with the language of beauty and expense ("beautiful", "pure tortoise shell", "jewelled rims", "expensive", "coveted adornments") at the exact moment when they cannot be used, because the hair they were bought for has been sold.

This is the main finding of the dimension: the strongest language of value and desire is attached to the combs when they have lost their practical use. The loss does not reduce the value of the gift in the language of the story. It increases it.

### Paradoxical value inversion: the narrator's voice

The three sub-types behave differently.

**Narrator comments (15 hits)** are not concentrated at the end. Stage I has 6 hits (1.21 per 100 words), the other stages 3 each. The final paragraph has 3 hits in 122 words (about 2.5 per 100 words), which is the highest density of any single segment. The narrator is therefore present at the start and at the end of the story, with a lower level in between. The comments also have different functions. Early comments, such as "Which is all very good" and the phrase "did not exactly beggar description", appear in the same segment as the shame vocabulary (segment 2) and make the poverty lighter in tone. The comments in the middle of the story ("Forget the hashed metaphor", "dear friends") speak directly to the reader. The comments at the end give the narrator's final verdict.

**Wisdom and foolishness (14 hits)** are almost entirely at the end: 11 of the 14 hits are in segment 13 ("wise men", "wonderfully wise men", "foolish children", "unwisely", "wisest", "magi"). The word "wise" does not occur before the last paragraph. Before it, the vocabulary of reason is used for moments of calculation and blankness: "prudence and reason" when Della calms down after shopping (segment 7), "an air almost of idiocy" for Jim after hearing about the hair (segment 9) and "A mathematician or a wit would give you the wrong answer" in the narrator's aside (segment 10). The reader is therefore first shown that calculation is not the right way to judge what has happened, and only in the last paragraph is told who is wise.

**Ironic register (14 hits)** is thin in Stage I (0.40 per 100 words) and stronger in Stages II to IV (0.86, 0.72 and 0.71). An example is segment 4, where the narrator compares Della's hair and Jim's watch with the jewels of the queen of Sheba and the treasures of King Solomon. The hair and the watch are raised above royal treasure through exaggeration, and this prepares the valuation stated openly at the end.

## 7.8 Testing the original claim

The original claim was: *words of shame, counting and selling are concentrated in the first two stages, while words of wisdom and the narrator's comments are concentrated in the last stage.*

| Part of the claim | Result | Evidence |
|---|---|---|
| Shame and counting words are early | **Supported, more strongly than claimed** | 29 of 37 shame hits and all 11 counting hits are in Stage I alone; shame falls steadily (rho = -0.81) |
| Selling words are in the first two stages | **Partly supported** | Selling and body vocabulary is highest in Stages II and III (3.08 and 2.88 per 100 words) and returns in Della's confession (segment 9) |
| Wisdom words are in the last stage | **Supported, with a qualification** | 11 of 14 hits are in a single paragraph (segment 13) |
| Narrator comments are in the last stage | **Not supported as stated** | They are highest in Stage I (6 hits) and again in the final paragraph (3 hits); the narrator is present at both ends |

## 7.9 Answer to the research question and return to the initial impression

The research question was: *How does the narrator's language turn economic loss into emotional value, and what does the reader learn to call "wise"?*

The initial impression, formed after the partial reading, was that money and love pull against each other and that the narrator's language turns loss into value. The results support this impression and show how it works. The conversion happens in four overlapping moves:

1. **Counting and shame come first.** The story opens in money (Stage I: 5.86 shame hits per 100 words) and in the embarrassment of scarcity.
2. **Numbers disappear.** Counting words are absent after Stage I and amounts of money are absent after Stage II.
3. **The sale of the hair is told in the language of goods and then in the language of confession.** The body is priced in segment 5 and the sale is spoken of again in segment 9.
4. **Feeling and value move onto the gifts.** The words of beauty, desire and joy are strongest around the combs (segment 11) and in the final stage (7.33 per 100 words), at the moment the gifts cannot be used.

**What the reader learns to call "wise".** The narrator names the result only in the last paragraph, where 11 of the 14 wisdom words appear. The reader is led to feel the value of the gifts before being told to call it wisdom, and the earlier comments about calculation prepare the final reversal: the two characters who gave up their most valuable possessions are called "the wisest".

The results also correct the first impression in two ways. The conversion is not one smooth process but several, running at different speeds. And the sale of the body is as prominent in the language as money, and returns in Stage III after the money language has gone.

## 7.10 Position of the reader and role of the model

The story was read by a college student from a busy, multilingual urban Indian setting, with English as an academic language rather than the language of the home. This position shaped what was noticed first: money, family obligation and feeling. Several of the most telling shame words, such as "parsimony", "mendicancy" and "meretricious", belong to a formal literary register that is easy to pass over when reading for plot and feeling. The extraction step made them visible, so the model complemented the reading rather than replacing it. The choice of the four dimensions, however, came from the reading itself. Another reader might begin from different concerns and arrive at a different set of dimensions.

## 7.11 Limitations

- **The method depends on a word list.** Words such as "present", "nice" and "hair" can have several senses, and counting treats all senses alike. One false positive ("just at present", meaning "now", not "gift") was caught with a stoplist, but others may remain.
- **The extraction was done by a language model** from a prompt written for this study. Every phrase was checked against the text, but the choice of which phrases to extract could differ for another model or another reader.
- **The division into 13 segments and 4 stages** was decided by the researcher from changes of scene. A different division would change the stage figures, although the strongest patterns (the fall of shaming and the final peak of inversion) are also visible in the individual segments.
- **The story is short.** A short segment, such as segment 5 with 80 words, can show a high density from a few hits.
- **Counts show vocabulary, not tone.** Irony in particular is only partly captured by phrases.
- **The statistical tests are a guide only**, because hits within a passage are not independent.

## 7.12 Conclusion

**Answer to the research question.** The narrator's language turns economic loss into emotional value gradually, in four steps, and teaches the reader to call the two characters "wise" only at the very end.

1. **Counting and shame come first.** The story opens in money: shame vocabulary is at 5.86 hits per 100 words in Stage I and falls steadily to 0.00 by Stage IV.
2. **The numbers disappear.** Counting words are absent after Stage I.
3. **The hair is sold in the language of goods and then of confession.** The sale (segment 5) and Della's confession (segment 9) are the two peaks of the corporeal vocabulary.
4. **Feeling and value move onto the gifts.** Words of beauty, desire and joy are strongest around the combs (segment 11) and in Stage IV (7.33 per 100 words), at the moment the gifts cannot be used.

**What the reader learns to call "wise".** The narrator names the result in the last paragraph, where 11 of the 14 wisdom words appear: the couple who sacrificed their most valuable possessions for each other are "the wisest" and "the magi". Earlier comments about calculation ("A mathematician or a wit would give you the wrong answer") prepare this reversal.

**Two additional findings.** The original claim is only partly supported: shame and counting are early, as expected, but selling language returns in Stage III and the narrator's comments appear at both the start and the end. The language of sacrifice is also unequal: most of the vocabulary of selling and of bodily reaction belongs to Della.

In [ ]:
def md_to_html(md_text):
    """Markdown converter: headings, paragraphs, bullet and numbered lists, tables, bold and italics."""
    def inline(t):
        t = escape(t)
        t = re.sub(r"\*\*(.+?)\*\*", r"<b>\1</b>", t)
        return re.sub(r"\*(.+?)\*", r"<i>\1</i>", t)
    out, mode, table = [], None, []

    def close():
        nonlocal mode
        if mode == "ul":
            out.append("</ul>")
        elif mode == "ol":
            out.append("</ol>")
        mode = None

    def flush_table():
        nonlocal table
        if table:
            rws = [[c.strip() for c in r.strip().strip("|").split("|")] for r in table]
            rws = [r for r in rws if not all(set(c) <= set("-: ") for c in r)]
            html = "<table>"
            for i, r in enumerate(rws):
                tag = "th" if i == 0 else "td"
                html += "<tr>" + "".join(f"<{tag}>{inline(c)}</{tag}>" for c in r) + "</tr>"
            out.append(html + "</table>")
            table = []

    for line in md_text.splitlines():
        if line.startswith("|"):
            close()
            table.append(line)
            continue
        flush_table()
        num = re.match(r"^\d+\.\s+(.*)", line)
        if line.startswith("- "):
            if mode != "ul":
                close(); out.append("<ul>"); mode = "ul"
            out.append(f"<li>{inline(line[2:])}</li>")
        elif num:
            if mode != "ol":
                close(); out.append("<ol>"); mode = "ol"
            out.append(f"<li>{inline(num.group(1))}</li>")
        else:
            close()
            if line.startswith("### "):   out.append(f"<h4>{inline(line[4:])}</h4>")
            elif line.startswith("## "):  out.append(f"<h3>{inline(line[3:])}</h3>")
            elif line.startswith("# "):   out.append(f"<h2>{inline(line[2:])}</h2>")
            elif line.strip():            out.append(f"<p>{inline(line)}</p>")
    close()
    flush_table()
    return "\n".join(out)

TITLE = "Reading “The Gift of the Magi” with an LLM: how language turns loss into value"
PAGE = """<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>@@TITLE@@</title>
<style>
body{font-family:Georgia,'Times New Roman',serif;color:#222;background:#fffdf8;max-width:920px;margin:0 auto;padding:2.5rem 1.2rem;line-height:1.55}
h1{font-size:1.8rem;line-height:1.25;color:#1f3a5f;margin-bottom:.4rem}
h2{font-size:1.25rem;margin-top:2.4rem;border-bottom:1px solid #d8d0bf;padding-bottom:.3rem;color:#1f3a5f}
h3{font-size:1.08rem;color:#1f3a5f;margin-bottom:.2rem}
section{margin:2.2rem 0;padding-bottom:1.2rem;border-bottom:1px solid #e3dccb}
.box{background:#f3eee4;border-left:4px solid #8c2f39;padding:.8rem 1.1rem;margin:1rem 0}
.how{color:#555;font-size:.95rem;margin:.2rem 0 .6rem}
.finding{background:#f3eee4;padding:.6rem .9rem;border-radius:4px}
table{border-collapse:collapse;width:100%;font-size:.9rem;margin:.6rem 0}
th{background:#1f3a5f;color:#fff;text-align:left;padding:.45rem}
td{border-bottom:1px solid #e3dccb;padding:.45rem}
small{color:#666}
@media (prefers-color-scheme: dark){body{background:#15171c;color:#e6e3dc}h1,h2,h3{color:#9db8e0}.box,.finding{background:#232730}td{border-color:#333}small,.how{color:#a9a9a9}}
</style></head><body>
<h1>@@TITLE@@</h1>
<small>Hover over any element for the phrases behind it. Use the toolbar at the top right of each chart to zoom or save a PNG.</small>
<div class="box"><b>Research question.</b> @@RQ@@<br><b>Claim tested.</b> @@CLAIM@@</div>
<h2>Visualisations</h2>
@@BODY@@
<h2>Results by stage (hits per 100 words)</h2>
@@STAGETABLE@@
<h2>How the story was divided</h2>
<p><small>Segments and stages were decided by the researcher from shifts of scene, not by equal length.</small></p>
@@SEGTABLE@@
@@ESSAY@@
<h2>Method note</h2>
<p><small>Phrases were extracted by @@MODEL@@ from a prompt written by the researcher, verified word for word against the text, and matched by whole words, longest phrase first, without counting any stretch of text twice. Tests are descriptive: hits within a passage are not independent.</small></p>
</body></html>"""

def build_page(path, with_essay):
    body = ""
    for k, f in enumerate(FIGS, 1):
        chart = f.to_html(full_html=False, include_plotlyjs="cdn" if k == 1 else False, config=CONFIG)
        body += (f"<section><h3>{escape(TITLES[k])}</h3><p class='how'><b>How to read this:</b> {escape(HOW[k])}</p>"
                 f"{chart}<p class='finding'><b>What it shows:</b> {escape(INTERPRETATION[k])}</p></section>\n")
    stage_table = stage_dens.rename(index=lambda d: DIMENSIONS[d]["label"]).round(2).to_html(border=0)
    seg_table = pd.DataFrame([{"Segment": s["id"], "Paragraphs": s["paras"], "Scene": s["note"],
                               "Stage": s["stage"], "Words": s["words"]} for s in segments]).to_html(index=False, border=0)
    page = (PAGE.replace("@@TITLE@@", escape(TITLE)).replace("@@RQ@@", escape(RESEARCH_QUESTION))
                .replace("@@CLAIM@@", escape(CLAIM)).replace("@@BODY@@", body)
                .replace("@@STAGETABLE@@", stage_table).replace("@@SEGTABLE@@", seg_table)
                .replace("@@MODEL@@", escape(MODEL_NAME))
                .replace("@@ESSAY@@", md_to_html(ESSAY) if with_essay else ""))
    open(path, "w", encoding="utf-8").write(page)
    print("Wrote", path)

build_page("docs/visualisations.html", with_essay=False)
build_page("docs/report.html", with_essay=True)

Wrote docs/visualisations.html
Wrote docs/report.html


In [ ]:
!zip -q -r synthetic_reading_gift_of_the_magi.zip corpus partial_reading prompt extraction docs initial_impression.md
try:
    from google.colab import files
    files.download("synthetic_reading_gift_of_the_magi.zip")
except ImportError:
    print("Not running in Colab: the zip file is in the working folder.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>